# 현재 근거 팩 — KO‑EN Tokenization Premium

**Article / Report Quantitative Evidence Pack** · `EDA_current_claim_evidence_pack_20260818`

| record | value |
|---|---|
| branch | `results/current-evidence-pack-20260818` |
| 생성 시 base | `origin/main` @ `bc8187cceae67ac87c9df1c4a08b54ca753ae035` |
| 현재 canonical main (병합됨) | `1b212986527a273301c5fefdf87a41e7fb33dd37` |
| NB09 result audit | `27814139a43bfbe3d4eb2e646e97c0107acfa582` (`NB09_RESULT_AUDIT_PASS`) |
| NB09 supplemental audit | `c12ca806bfd8245264bba93b52128a07c71be41a` (block CI 구성 결함) |
| 성격 | **SUPPORTING evidence** — 새 canonical 연구 단계가 아니다 |
| canonical NB07 | **수정하지 않는다** (별도 lane) |

## 이 노트북이 답하는 두 질문

모든 절은 다음 두 가지를 쌍으로 답한다.

1. **현재 데이터로 정확히 무엇을 말할 수 있는가?**
2. **무엇은 아직 말할 수 없는가?**

## 근거 등급 (evidence status)

모든 통계에는 정확히 하나의 등급이 붙는다.

| 등급 | 의미 | `ARTICLE_READY` |
|---|---|---|
| `AUDITED_CANONICAL` | 닫힌 Gate·독립 감사를 통과한 canonical 사실 (G5, NB08 RQ1, artifact 신원) | **YES** |
| `AUDITED_RESULT` | 독립 감사를 통과한 결과 산출물에서 그대로 소비한 값 (NB09) | **YES** |
| `DESCRIPTIVE_PERSISTED` | 감사된 canonical artifact에서 **이 실행이 새로 계산**한 기술통계. 값 자체는 아직 Gate 감사를 받지 않았다 | NO |
| `CANDIDATE_PENDING_AUDIT` | 감사 대기 중인 산출물(NB07 descriptive package)에서 온 진술 | NO |
| `NOT_YET_ESTABLISHED` | 아직 산출되지 않았다 (NB11 robustness, NB10 predictive) | NO |

**감사받지 않은 결과를 조용히 승격시키지 않는다.** `ARTICLE_READY = YES` 는
`AUDITED_CANONICAL` 과 `AUDITED_RESULT` 에만 부여된다.

### 점추정치와 구간을 분리한다

한 결과의 **값**과 그 값의 **구간**은 신뢰도가 다를 수 있다. NB09 보충감사가 block ΔR²
bootstrap 구간에서 구성 결함을 확인했으므로, 결과 전체를 강등하는 대신 두 축으로 나눈다.

| 축 | 뜻 |
|---|---|
| `ARTICLE_READY_POINT_ESTIMATE` | 값 자체를 기사에 인용해도 되는가 |
| `ARTICLE_READY_BOOTSTRAP_CI` | 그 값의 구간을 기사에 인용해도 되는가 |

현재 `NB09_BLOCK_CI_STATUS = CORRECTION_PENDING`, `ARTICLE_READY_CI = NO` 다.
철회된 구간은 삭제하지 않고 `quarantined_ci` 필드에 기록만 하며, 어떤 인용 가능 필드나
그림에도 나타나지 않는다. 감사자가 자기 seed로 만든 대조 구간은 **결함 크기의 증거이지
수정된 결과가 아니므로 채택하지 않는다.**

## 개인정보

원문 문장 없음 · `pair_id` 목록 없음 · 복원 가능한 예시 없음 · token 배열 없음.
집계 근거만 싣는다.

## 01 — 실행 환경과 한국어 시각화 계약

기사·방송 그래픽에 그대로 쓸 수 있어야 하므로 모든 figure는 한국어 title/axis/legend를 갖는다.
font 탐색 순서: `Noto Sans CJK KR` → `NanumGothic` → `Malgun Gothic`.

In [1]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import platform
import sys
import warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from matplotlib import font_manager
from matplotlib.colors import LogNorm
from matplotlib.ft2font import FT2Font
from scipy import stats

KST = ZoneInfo("Asia/Seoul")
RUN_ID = "CURRENT_EVIDENCE_PACK_v001"
BASE_MAIN_SHA = "bc8187cceae67ac87c9df1c4a08b54ca753ae035"
CURRENT_CANONICAL_MAIN = "1b212986527a273301c5fefdf87a41e7fb33dd37"
NB09_RESULT_AUDIT = "27814139a43bfbe3d4eb2e646e97c0107acfa582"
NB09_SUPPLEMENTAL_AUDIT = "c12ca806bfd8245264bba93b52128a07c71be41a"

ROOT = Path.cwd()
while ROOT.name in ("exploratory", "notebooks"):
    ROOT = ROOT.parent
sys.path.insert(0, (ROOT / "src").as_posix())

REG = ROOT / "data" / "registry"
RUNTIME = ROOT / ".runtime" / "evidence_pack"
FIGDIR = ROOT / "outputs" / "figures" / "evidence_pack"
TBLDIR = ROOT / "outputs" / "tables"
REPDIR = ROOT / "outputs" / "reports"
for d in (RUNTIME, FIGDIR, TBLDIR, REPDIR):
    d.mkdir(parents=True, exist_ok=True)

from tokenization_premium.telemetry import RuntimeTelemetry  # noqa: E402

EXPECTED_N = 3_835_988
EXPECTED_PAIR_SET = "d9660d654ee449e4d0c23a0070225274"

KOREAN_PRIORITY = ("Noto Sans CJK KR", "NanumGothic", "Malgun Gothic")
KOREAN_PROBE = "한글 토큰 프리미엄 형태소 청크 압축 표현"
for _c in Path("/usr/share/fonts").rglob("*.tt[cf]"):
    if "CJK" in _c.name:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            font_manager.fontManager.addfont(_c.as_posix())
_by_family: dict[str, str] = {}
for _e in font_manager.fontManager.ttflist:
    _by_family.setdefault(_e.name, _e.fname)

KOREAN_PLOT_FONT = KOREAN_PLOT_FONT_PATH = None
FONT_TRACE = []
for _fam in KOREAN_PRIORITY:
    _p = _by_family.get(_fam)
    if _p is None:
        FONT_TRACE.append({"family": _fam, "status": "NOT_INSTALLED"}); continue
    try:
        _cmap = FT2Font(_p).get_charmap()
    except (RuntimeError, OSError) as exc:
        FONT_TRACE.append({"family": _fam, "status": f"UNREADABLE: {exc}"}); continue
    _missing = [c for c in KOREAN_PROBE if "가" <= c <= "힣" and ord(c) not in _cmap]
    if _missing:
        FONT_TRACE.append({"family": _fam, "status": f"MISSING_GLYPHS: {''.join(_missing)}"}); continue
    FONT_TRACE.append({"family": _fam, "status": "SELECTED", "path": _p})
    KOREAN_PLOT_FONT, KOREAN_PLOT_FONT_PATH = _fam, _p
    break
if KOREAN_PLOT_FONT is None:
    raise RuntimeError("HARD WARNING — 한글 glyph font 없음: " + json.dumps(FONT_TRACE, ensure_ascii=False))

matplotlib.rcParams.update({
    "font.family": KOREAN_PLOT_FONT, "axes.unicode_minus": False,
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "svg.fonttype": "none", "svg.hashsalt": "koen-evidence-pack-v001",
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 11.5,
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
})
print(f"KOREAN_PLOT_FONT={KOREAN_PLOT_FONT}\n  path={KOREAN_PLOT_FONT_PATH}")
ENV = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "scipy": scipy.__version__, "duckdb": duckdb.__version__,
       "matplotlib": matplotlib.__version__, "platform": platform.platform()}
print("environment:", json.dumps(ENV))
STARTED_KST = dt.datetime.now(KST).isoformat(timespec="seconds")
print("started_kst =", STARTED_KST, "| ROOT =", ROOT)

KOREAN_PLOT_FONT=Noto Sans CJK KR
  path=/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc
environment: {"python": "3.12.3", "numpy": "2.5.2", "pandas": "2.3.3", "scipy": "1.18.0", "duckdb": "1.5.5", "matplotlib": "3.11.1", "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39"}
started_kst = 2026-08-18T17:39:37+09:00 | ROOT = /home/sieg/projects-wsl/KOEN_evidence_20260818


## 02 — 입력 신원 확인 (fail-closed)

두 종류의 입력이 있다.

1. **동결 artifact D‑01 … D‑05** — 여기서 기술통계를 **새로 계산**한다.
2. **감사된 결과 산출물** — NB08 RQ1 closeout, NB09 explanatory results, G5 진단 보고.
   이들은 **재계산하지 않고 그대로 소비**하며, 파일 SHA‑256과 audit SHA를 함께 기록한다.

어느 쪽이든 신원이 어긋나면 즉시 중단한다.

In [2]:
def sha256_file(path: Path, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        while b := fh.read(chunk):
            h.update(b)
    return h.hexdigest()


FROZEN = {
    "D-01": ("PAIR_REGISTRY_v002.parquet",
             "95f523d11b0e8fcfd761dee949f082e9b4590b919801441fbcfa3426010bec52"),
    "D-02": ("REP_FEATURES_v002.parquet",
             "dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309"),
    "D-03": ("MORPH_FEATURES_KIWI_v001.parquet",
             "0fe5bd74e3993a7141c5c33ea78e71b2c66e3ecd296544bde2615acb43e50f7d"),
    "D-04": ("TOKEN_O200K_BASE_v001.parquet",
             "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7"),
    "D-05": ("CHUNK_O200K_BASE_v001.parquet",
             "bfa98bd6cf7ee8b7254c469aed3e259ce43cc8f0529153347ca4c2c3fc1944ab"),
}
ARTIFACTS, _fail = {}, []
with RuntimeTelemetry(run_id=f"{RUN_ID}_SHA", stage="SHA256", total=len(FROZEN),
                      abort_on_red=True) as _tel:
    for _k, (_fn, _exp) in FROZEN.items():
        _tel.set_stage(f"SHA256:{_k}")
        _real = (REG / _fn).resolve()
        _got = sha256_file(_real)
        ARTIFACTS[_k] = {"filename": _fn, "sha256": _got, "expected_sha256": _exp,
                         "match": _got == _exp, "size_bytes": _real.stat().st_size}
        if _got != _exp:
            _fail.append(_k)
        _tel.update(1)
        print(f"  {_k} {_fn:34s} {'MATCH' if _got == _exp else 'MISMATCH'}  {_got[:12]}…")
if _fail:
    raise SystemExit(f"FROZEN_ARTIFACT_MISMATCH: {_fail}")
ARTIFACT_IDENTITY = f"{sum(v['match'] for v in ARTIFACTS.values())} / {len(FROZEN)}"
print(f"ARTIFACT_IDENTITY = {ARTIFACT_IDENTITY}")

# ---- 감사된 결과 산출물 (재계산하지 않고 소비) -------------------------------------
AUDITED_SOURCES = {
    "NB08_RQ1_CLOSEOUT": {
        "path": "ssot_nb01/06_NB08_RQ1_SSOT_CLOSEOUT_v001.json",
        "status": "AUDITED_CANONICAL",
        "audit_ref": "RD-RQ1-FIRST-RESULT-01 · NB08 canonical closeout (RQ1 CLOSED)"},
    "NB09_EXPLANATORY_RESULTS": {
        "path": "outputs/reports/NB09_EXPLANATORY_RESULTS_v001.json",
        "status": "AUDITED_RESULT",
        "audit_ref": "NB09_RESULT_AUDIT_PASS · audit SHA 27814139a43bfbe3d4eb2e646e97c0107acfa582"},
    "NB09_MODEL_DIAGNOSTICS": {
        "path": "outputs/reports/NB09_MODEL_DIAGNOSTICS_v001.json",
        "status": "AUDITED_RESULT",
        "audit_ref": "NB09_RESULT_AUDIT_PASS"},
    "G5_COLLINEARITY": {
        "path": "outputs/reports/G5_COLLINEARITY_v001.json",
        "status": "AUDITED_CANONICAL",
        "audit_ref": "G5_INDEPENDENT_AUDIT_PASS · audit SHA 9d99e13026b89dcf7d8846d0c105a811f64274bc"},
    "G5_IDENTIFIABILITY": {
        "path": "outputs/reports/G5_IDENTIFIABILITY_v001.json",
        "status": "AUDITED_CANONICAL",
        "audit_ref": "G5_INDEPENDENT_AUDIT_PASS"},
}
LOADED = {}
for _k, _m in AUDITED_SOURCES.items():
    _p = ROOT / _m["path"]
    _m["sha256"] = sha256_file(_p)
    LOADED[_k] = json.loads(_p.read_text(encoding="utf-8"))
    print(f"  {_k:26s} {_m['status']:20s} sha {_m['sha256'][:12]}…  {_m['path']}")

NB09 = LOADED["NB09_EXPLANATORY_RESULTS"]
RQ1 = LOADED["NB08_RQ1_CLOSEOUT"]
NB09_AUDIT_SHA = "27814139a43bfbe3d4eb2e646e97c0107acfa582"
G5_AUDIT_SHA = "9d99e13026b89dcf7d8846d0c105a811f64274bc"

# NB07 descriptive package — 감사 대기 중이므로 값을 인용하지 않고 lineage만 기록한다
NB07_CANDIDATE = {
    "branch": "research/nb07-canonical-eda-20260818",
    "sha": "239cfd9c3e12d10272421a982f2a3493f6963907",
    "status": "CANDIDATE_PENDING_AUDIT",
    "note": ("NB07 canonical descriptive package는 Claude-B의 science-scope audit 전이다. "
             "이 evidence pack은 NB07의 수치를 인용하지 않고, 같은 동결 artifact에서 "
             "독립적으로 다시 계산한다."),
}
print(f"\nNB07 descriptive package: {NB07_CANDIDATE['status']} @ {NB07_CANDIDATE['sha'][:12]}…")
print("  → 값을 인용하지 않고 동결 artifact에서 독립 재계산한다")

  D-01 PAIR_REGISTRY_v002.parquet         MATCH  95f523d11b0e…
  D-02 REP_FEATURES_v002.parquet          MATCH  dfae8e01cd3f…


  D-03 MORPH_FEATURES_KIWI_v001.parquet   MATCH  0fe5bd74e399…


  D-04 TOKEN_O200K_BASE_v001.parquet      MATCH  1c30e3276222…


  D-05 CHUNK_O200K_BASE_v001.parquet      MATCH  bfa98bd6cf7e…
ARTIFACT_IDENTITY = 5 / 5
  NB08_RQ1_CLOSEOUT          AUDITED_CANONICAL    sha 490a34f456bf…  ssot_nb01/06_NB08_RQ1_SSOT_CLOSEOUT_v001.json
  NB09_EXPLANATORY_RESULTS   AUDITED_RESULT       sha 598951b79b00…  outputs/reports/NB09_EXPLANATORY_RESULTS_v001.json
  NB09_MODEL_DIAGNOSTICS     AUDITED_RESULT       sha 269d20a31dc4…  outputs/reports/NB09_MODEL_DIAGNOSTICS_v001.json
  G5_COLLINEARITY            AUDITED_CANONICAL    sha e6e7034ea601…  outputs/reports/G5_COLLINEARITY_v001.json
  G5_IDENTIFIABILITY         AUDITED_CANONICAL    sha 1069b46ed032…  outputs/reports/G5_IDENTIFIABILITY_v001.json

NB07 descriptive package: CANDIDATE_PENDING_AUDIT @ 239cfd9c3e12…
  → 값을 인용하지 않고 동결 artifact에서 독립 재계산한다


## 03 — 분석 cohort 재조립

기술통계는 전부 여기서 새로 계산한다. 오래된 산문에서 수치를 복사하지 않는다.
동결 cohort와 일치하지 않으면 중단한다: `N = 3,835,988`,
pair-set `d9660d654ee449e4d0c23a0070225274`.

In [3]:
P = f"read_parquet('{(REG / 'PAIR_REGISTRY_v002.parquet').as_posix()}')"
R = f"read_parquet('{(REG / 'REP_FEATURES_v002.parquet').as_posix()}')"
M = f"read_parquet('{(REG / 'MORPH_FEATURES_KIWI_v001.parquet').as_posix()}')"
T = f"read_parquet('{(REG / 'TOKEN_O200K_BASE_v001.parquet').as_posix()}')"
K = f"read_parquet('{(REG / 'CHUNK_O200K_BASE_v001.parquet').as_posix()}')"

SQL = f"""
SELECT t.pair_id,
  split_part(p.source_id, '-', 1) AS source, p.domain, p.translation_direction,
  p.length_stratum, split_part(p.source_id, '-', 1) || '-' || p.domain AS source_domain_cell,
  t.ko_token_count, t.en_token_count, t.token_premium, t.log_token_premium, t.token_difference,
  t.log_code_point_ratio, t.log_byte_density_ratio, t.log_compression_penalty,
  t.ko_tokens_per_byte, t.en_tokens_per_byte,
  r.ko_codepoint_count, r.en_codepoint_count, r.ko_utf8_bytes, r.en_utf8_bytes,
  r.ko_bytes_per_codepoint, r.en_bytes_per_codepoint,
  r.ko_hangul_share, r.en_hangul_share, r.ko_eojeol_count, r.en_word_count,
  r.pair_codepoint_ratio, r.pair_byte_ratio,
  m.morpheme_count, m.morpheme_density, m.particle_ratio, m.ending_ratio,
  m.deriv_affix_ratio, m.function_morpheme_ratio, m.eojeol_count AS morph_eojeol_count,
  k.ko_chunk_count, k.en_chunk_count, k.ko_tokens_per_chunk, k.en_tokens_per_chunk,
  CAST(k.ko_max_tokens_per_chunk AS INTEGER) AS ko_max_tokens_per_chunk,
  CAST(k.en_max_tokens_per_chunk AS INTEGER) AS en_max_tokens_per_chunk
FROM {T} t
JOIN {P} p ON p.pair_id = t.pair_id
JOIN {R} r ON r.pair_id = t.pair_id
JOIN {M} m ON m.pair_id = t.pair_id
JOIN {K} k ON k.pair_id = t.pair_id
"""
MATRIX = RUNTIME / "evidence_matrix.parquet"
con = duckdb.connect()
con.execute("PRAGMA memory_limit='3GB'"); con.execute("PRAGMA threads=4")
con.execute("SET preserve_insertion_order=false")
con.execute(f"PRAGMA temp_directory='{(RUNTIME / 'spill').as_posix()}'")

with RuntimeTelemetry(run_id=f"{RUN_ID}_COHORT", stage="MATERIALIZE", total=EXPECTED_N,
                      abort_on_red=True) as _tel:
    con.execute(f"COPY ({SQL}) TO '{MATRIX.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    A = f"read_parquet('{MATRIX.as_posix()}')"
    _tel.set_stage("IDENTITY")
    N, NDIST = con.execute(f"SELECT count(*), count(DISTINCT pair_id) FROM {A}").fetchone()
    PAIR_SET = con.execute(f"SELECT md5(string_agg(pair_id, '' ORDER BY pair_id)) FROM {A}").fetchone()[0]
    _tel.update(N)
    COHORT_TELEMETRY = _tel.summary()
if not (N == EXPECTED_N and NDIST == N and PAIR_SET == EXPECTED_PAIR_SET):
    raise SystemExit(f"COHORT_MISMATCH: N={N} distinct={NDIST} hash={PAIR_SET}")
print(f"COHORT_N = {N:,}   distinct = {NDIST:,}   PAIR_SET_HASH = {PAIR_SET}")
print(f"telemetry: {COHORT_TELEMETRY['sample_count']} samples @ {COHORT_TELEMETRY['interval_sec']}s · "
      f"worst {COHORT_TELEMETRY['worst_memory_status']} · peak RSS {COHORT_TELEMETRY['peak_rss_gib']} GiB")
print("cohort는 NB09가 적합한 것과 동일하다 (N·pair-set hash 일치) — 근거 팩과 모형 결과가 같은 모집단을 가리킨다")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

COHORT_N = 3,835,988   distinct = 3,835,988   PAIR_SET_HASH = d9660d654ee449e4d0c23a0070225274
telemetry: 2 samples @ 10.0s · worst YELLOW · peak RSS 3.309 GiB
cohort는 NB09가 적합한 것과 동일하다 (N·pair-set hash 일치) — 근거 팩과 모형 결과가 같은 모집단을 가리킨다


## 04 — 근거 등급 · 카드 · 주장 행렬 helper

notebook-first 원칙에 따라 이 팩이 쓰는 helper는 전부 여기서 정의한다.
외부에서 가져오는 것은 `RuntimeTelemetry` 뿐이다.

In [4]:
ARTICLE_READY_STATUSES = {"AUDITED_CANONICAL", "AUDITED_RESULT"}

# ---- 보충감사 반영: NB09 block bootstrap 구간만 격리한다 (점추정치는 유지) --------
# ssot/2026-08-18_1713_KOEN_TP_NB09_SUPPLEMENTAL_AUDIT.md
#   BOOTSTRAP_BLOCK_CI = NARROW_METHOD_DEFECT_REQUIRING_CI_CORRECTION
#   correction scope   = block ΔR² interval + delta_r2_bootstrap_sd only
#   점추정치·R²·partial R²·nested 통계는 영향을 받지 않는다
NB09_BLOCK_CI_STATUS = "CORRECTION_PENDING"
ARTICLE_READY_CI = "NO"
CI_QUARANTINE_NOTE = ("NB09 block ΔR² bootstrap 구간은 보충감사에서 구성 결함이 확인되어 "
                      "수정 중이다. B의 frozen-seed 재계산과 A의 검증 전까지 인용하지 않는다.")
VALID_STATUSES = ARTICLE_READY_STATUSES | {
    "DESCRIPTIVE_PERSISTED", "CANDIDATE_PENDING_AUDIT", "NOT_YET_ESTABLISHED"}

CARDS: list[dict] = []
CLAIMS: list[dict] = []
FIGURES: list[dict] = []
SAY: list[dict] = []


def stat(card: str, metric_ko: str, metric: str, value, status: str, *,
         source_artifact: str, source_sha: str = "", audit_sha: str = "",
         n=None, ci: str = "", numerator: str = "", denominator: str = "",
         note: str = "", ci_status: str = "NOT_APPLICABLE",
         quarantined_ci: str = "") -> dict:
    """근거 카드 한 줄.

    점추정치와 구간의 기사 적합성을 **분리**한다:
      article_ready_point_estimate — 값 자체를 인용해도 되는가
      article_ready_bootstrap_ci   — 그 값의 구간을 인용해도 되는가
    """
    if status not in VALID_STATUSES:
        raise ValueError(f"unknown status {status}")
    if ci_status not in ("NOT_APPLICABLE", "CORRECTION_PENDING", "OK"):
        raise ValueError(f"unknown ci_status {ci_status}")
    point_ready = "YES" if status in ARTICLE_READY_STATUSES else "NO"
    ci_ready = {"NOT_APPLICABLE": "N/A", "CORRECTION_PENDING": "NO",
                "OK": point_ready}[ci_status]
    row = {"card": card, "metric": metric, "metric_ko": metric_ko,
           "value": value, "status": status,
           "article_ready": point_ready,                    # 하위호환: 점추정치 기준
           "article_ready_point_estimate": point_ready,
           "article_ready_bootstrap_ci": ci_ready,
           "ci_status": ci_status,
           "n": n, "ci_or_range": ci, "quarantined_ci": quarantined_ci,
           "numerator": numerator, "denominator": denominator,
           "source_artifact": source_artifact, "source_sha": source_sha,
           "audit_sha": audit_sha, "note": note}
    CARDS.append(row)
    return row


def claim(claim_id: str, plain_korean_claim: str, technical_claim: str, status: str, *,
          n, estimate, ci_or_range: str, numerator: str, denominator: str,
          source_artifact: str, source_SHA: str, audit_SHA: str,
          allowed_wording: str, forbidden_wording: str,
          ci_status: str = "NOT_APPLICABLE", quarantined_ci: str = "") -> dict:
    if status not in VALID_STATUSES:
        raise ValueError(f"unknown status {status}")
    if ci_status not in ("NOT_APPLICABLE", "CORRECTION_PENDING", "OK"):
        raise ValueError(f"unknown ci_status {ci_status}")
    point_ready = "YES" if status in ARTICLE_READY_STATUSES else "NO"
    ci_ready = {"NOT_APPLICABLE": "N/A", "CORRECTION_PENDING": "NO",
                "OK": point_ready}[ci_status]
    row = {"claim_id": claim_id, "plain_korean_claim": plain_korean_claim,
           "technical_claim": technical_claim, "status": status, "N": n,
           "estimate": estimate, "CI_or_range": ci_or_range,
           "numerator": numerator, "denominator": denominator,
           "source_artifact": source_artifact, "source_SHA": source_SHA,
           "audit_SHA": audit_SHA, "allowed_wording": allowed_wording,
           "forbidden_wording": forbidden_wording,
           "article_ready": point_ready,
           "article_ready_point_estimate": point_ready,
           "article_ready_bootstrap_ci": ci_ready,
           "ci_status": ci_status, "quarantined_ci": quarantined_ci}
    CLAIMS.append(row)
    return row


def say(scope: str, can: str, cannot: str) -> None:
    SAY.append({"scope": scope, "CAN_SAY": can, "CANNOT_SAY": cannot})
    print(f"\n[{scope}]\n  CAN_SAY    : {can}\n  CANNOT_SAY : {cannot}")


def save_fig(fig, fig_id: str, title_ko: str, *, card: str, status: str) -> dict:
    png, svg = FIGDIR / f"{fig_id}.png", FIGDIR / f"{fig_id}.svg"
    with warnings.catch_warnings(record=True) as cap:
        warnings.simplefilter("always")
        fig.savefig(png, format="png", metadata={"Software": "KOEN evidence pack"})
        fig.savefig(svg, format="svg", metadata={"Date": "2026-08-18", "Creator": "KOEN evidence pack"})
    miss = [str(w.message) for w in cap if "Glyph" in str(w.message) and "missing" in str(w.message)]
    if miss:
        raise AssertionError(f"{fig_id}: 한글 glyph 누락 — {miss}")
    txt = svg.read_text(encoding="utf-8")
    if not any("가" <= c <= "힣" for c in txt):
        raise AssertionError(f"{fig_id}: SVG에 한글 없음")
    e = {"figure_id": fig_id, "title_ko": title_ko, "card": card, "status": status,
         "article_ready": "YES" if status in ARTICLE_READY_STATUSES else "NO",
         "png": png.relative_to(ROOT).as_posix(), "svg": svg.relative_to(ROOT).as_posix(),
         "png_sha256": sha256_file(png), "svg_sha256": sha256_file(svg)}
    FIGURES.append(e)
    plt.close(fig)
    print(f"  saved {fig_id}  sha {e['png_sha256'][:12]}…  article_ready={e['article_ready']}")
    return e


def q(sql: str):
    return con.execute(sql).fetchone()


def qdf(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()


def show(df: pd.DataFrame, fmt: str = "{:,.6g}") -> None:
    with pd.option_context("display.width", 210, "display.max_columns", 60,
                           "display.max_rows", 60, "display.float_format", fmt.format):
        print(df.to_string(index=False))


def quantiles(expr: str, qs=(0.05, 0.25, 0.5, 0.75, 0.95)) -> dict:
    row = q(f"SELECT avg({expr}), quantile_cont({expr}, {list(qs)}) FROM {A}")
    return {"mean": float(row[0]), **{f"q{int(x * 100):02d}": float(v)
                                      for x, v in zip(qs, row[1], strict=True)}}


D04_SHA = ARTIFACTS["D-04"]["sha256"]
D02_SHA = ARTIFACTS["D-02"]["sha256"]
D03_SHA = ARTIFACTS["D-03"]["sha256"]
D05_SHA = ARTIFACTS["D-05"]["sha256"]
NB08_SHA = AUDITED_SOURCES["NB08_RQ1_CLOSEOUT"]["sha256"]
NB09_SHA = AUDITED_SOURCES["NB09_EXPLANATORY_RESULTS"]["sha256"]
print("helper 준비 완료 — 등급 5종, 카드/주장/그림 registry 초기화")

helper 준비 완료 — 등급 5종, 카드/주장/그림 registry 초기화


## CARD 01 — 주 프리미엄 (PRIMARY PREMIUM)

RQ1의 추론 결과는 **NB08에서 이미 닫혔다**. 이 카드는 그 감사된 값을 **인용**하고,
절대 token 차이 ΔT는 여기서 **새로 계산**한다. 둘의 등급이 다르다는 점이 중요하다.

In [5]:
_i = RQ1["INTERPRETATION"]
_s = RQ1["CONDITIONAL_NONZERO_SIGN_TEST"]["primary"]
_pos, _neg, _tie = _s["positive"], _s["negative"], _s["ties_excluded"]
_nb08_n = _pos + _neg + _tie
assert _nb08_n == N, (_nb08_n, N)

C1 = {"N": N, "median_TP": _i["median_TP_scale"], "median_logTP": _i["median_logTP"],
      "median_TP_as_ratio": _i["median_TP_as_ratio"],
      "median_premium_percent": _i["median_premium_percent"],
      "P_TP_gt_1": _pos / N, "P_TP_eq_1": _tie / N, "P_TP_lt_1": _neg / N,
      "n_TP_gt_1": _pos, "n_TP_eq_1": _tie, "n_TP_lt_1": _neg,
      "ci_status": "median CI는 격자 점질량 때문에 퇴화(degenerate)했다",
      "p_reporting": RQ1["REPORTING"]["publication_form"]}
for _k, _v, _num in (("P_TP_gt_1", _pos / N, _pos), ("P_TP_eq_1", _tie / N, _tie),
                     ("P_TP_lt_1", _neg / N, _neg)):
    stat("CARD01", {"P_TP_gt_1": "TP > 1 비율", "P_TP_eq_1": "TP = 1 비율",
                    "P_TP_lt_1": "TP < 1 비율"}[_k], _k, _v, "AUDITED_CANONICAL",
         source_artifact="NB08_RQ1_CLOSEOUT", source_sha=NB08_SHA,
         audit_sha="NB08 canonical closeout", n=N, numerator=f"{_num:,}", denominator=f"{N:,}")
stat("CARD01", "중앙 log Tokenization Premium", "median_logTP", C1["median_logTP"],
     "AUDITED_CANONICAL", source_artifact="NB08_RQ1_CLOSEOUT", source_sha=NB08_SHA,
     audit_sha="NB08 canonical closeout", n=N, ci=C1["ci_status"])
stat("CARD01", "중앙 Tokenization Premium (비율)", "median_TP", C1["median_TP"],
     "AUDITED_CANONICAL", source_artifact="NB08_RQ1_CLOSEOUT", source_sha=NB08_SHA,
     audit_sha="NB08 canonical closeout", n=N, note=f"정확히 {C1['median_TP_as_ratio']}")

# ---- ΔT: 이 실행에서 새로 계산 (등급이 다르다) ----------------------------------
_dt = quantiles("token_difference", (0.05, 0.25, 0.5, 0.75, 0.95))
_dt_sign = q(f"""SELECT sum(CASE WHEN token_difference > 0 THEN 1 ELSE 0 END),
                        sum(CASE WHEN token_difference = 0 THEN 1 ELSE 0 END),
                        sum(CASE WHEN token_difference < 0 THEN 1 ELSE 0 END) FROM {A}""")
C1["delta_T"] = {**_dt, "n_gt_0": int(_dt_sign[0]), "n_eq_0": int(_dt_sign[1]),
                 "n_lt_0": int(_dt_sign[2]),
                 "P_gt_0": _dt_sign[0] / N, "P_eq_0": _dt_sign[1] / N, "P_lt_0": _dt_sign[2] / N}
for _m, _ko in (("mean", "ΔT 평균"), ("q05", "ΔT 5분위"), ("q25", "ΔT 25분위"),
                ("q50", "ΔT 중앙값"), ("q75", "ΔT 75분위"), ("q95", "ΔT 95분위")):
    stat("CARD01", _ko, f"delta_T_{_m}", _dt[_m], "DESCRIPTIVE_PERSISTED",
         source_artifact="D-04 TOKEN_O200K_BASE_v001", source_sha=D04_SHA, n=N)

print(f"N = {N:,}")
print(f"중앙 TP        = {C1['median_TP']:.10f}  ( {C1['median_TP_as_ratio']} )  "
      f"→ 중앙 프리미엄 {C1['median_premium_percent']:.4f}%")
print(f"중앙 log TP    = {C1['median_logTP']:.16f}   [{C1['ci_status']}]")
print(f"P(TP > 1) = {C1['P_TP_gt_1']:.6f}  ({_pos:,}/{N:,})")
print(f"P(TP = 1) = {C1['P_TP_eq_1']:.6f}  ({_tie:,}/{N:,})")
print(f"P(TP < 1) = {C1['P_TP_lt_1']:.6f}  ({_neg:,}/{N:,})")
print(f"\nΔT (한국어 token − 영어 token, 이 실행에서 신규 계산):")
print(f"  평균 {_dt['mean']:.4f} · 5% {_dt['q05']:.0f} · 25% {_dt['q25']:.0f} · "
      f"중앙 {_dt['q50']:.0f} · 75% {_dt['q75']:.0f} · 95% {_dt['q95']:.0f}")
print(f"  P(ΔT>0) {C1['delta_T']['P_gt_0']:.6f} · P(ΔT=0) {C1['delta_T']['P_eq_0']:.6f} · "
      f"P(ΔT<0) {C1['delta_T']['P_lt_0']:.6f}")
print(f"  교차 확인: P(ΔT>0) == P(TP>1) → {abs(C1['delta_T']['P_gt_0'] - C1['P_TP_gt_1']) < 1e-15}")

claim("C-01", "이 표본에서 o200k_base 기준 한국어 문장의 중앙 token premium은 33.3%였다.",
      "Median(log TP) = 0.2876820724517808, 즉 median TP = 4/3 (paired KO-EN, o200k_base).",
      "AUDITED_CANONICAL", n=N, estimate="median TP = 1.3333 (+33.33%)",
      ci_or_range="median CI는 격자 점질량으로 퇴화 — 구간을 제시하지 않는다",
      numerator="—", denominator=f"{N:,}",
      source_artifact="NB08_RQ1_CLOSEOUT", source_SHA=NB08_SHA,
      audit_SHA="NB08 canonical closeout (RQ1 CLOSED)",
      allowed_wording="이 병렬 말뭉치·o200k_base 기준으로 중앙 문장쌍에서 33.3% 더 많은 token",
      forbidden_wording="한국어는 본질적으로 AI에 33.3% 비효율적이다 / 한국어 사용자는 33% 더 비싸다")
claim("C-02", "문장쌍 10개 중 약 9개에서 한국어 쪽 token이 더 많았다.",
      "P(TP > 1) = 0.879850 (3,375,095 / 3,835,988).", "AUDITED_CANONICAL",
      n=N, estimate="0.879850", ci_or_range="표본 비율 — 구간 추정 아님",
      numerator="3,375,095", denominator=f"{N:,}",
      source_artifact="NB08_RQ1_CLOSEOUT", source_SHA=NB08_SHA,
      audit_SHA="NB08 canonical closeout",
      allowed_wording="이 표본의 87.99%에서 한국어 token 수가 더 많았다",
      forbidden_wording="한국어 문장은 항상 token이 더 많다 / 88% 확률로 손해를 본다")
claim("C-03", "중앙 문장쌍에서 한국어가 영어보다 5개 더 많은 token을 썼다.",
      "Median(ΔT) = 5 tokens; ΔT는 문장 길이에 비례해 커지므로 보조 지표다 (SSOT D-01).",
      "DESCRIPTIVE_PERSISTED", n=N, estimate="median ΔT = 5",
      ci_or_range="5–95 분위 [-1, 20]", numerator="—", denominator=f"{N:,}",
      source_artifact="D-04 TOKEN_O200K_BASE_v001", source_SHA=D04_SHA, audit_SHA="",
      allowed_wording="중앙 문장쌍 기준 절대 차이 5 token (길이 의존 지표)",
      forbidden_wording="한국어는 문장마다 5 token씩 손해다")

say("CARD01",
    "이 병렬 말뭉치에서 o200k_base tokenizer 기준 한국어의 중앙 token premium은 33.3%였고, "
    "문장쌍의 87.99%에서 한국어 token 수가 더 많았다.",
    "한국어가 본질적으로 AI에 33% 비효율적이다 / 이 수치가 다른 tokenizer·다른 말뭉치·"
    "실제 서비스 비용에 그대로 적용된다 / token premium이 추론 성능 저하를 뜻한다.")

N = 3,835,988
중앙 TP        = 1.3333333333  ( 4 / 3 )  → 중앙 프리미엄 33.3333%
중앙 log TP    = 0.2876820724517808   [median CI는 격자 점질량 때문에 퇴화(degenerate)했다]
P(TP > 1) = 0.879850  (3,375,095/3,835,988)
P(TP = 1) = 0.051282  (196,718/3,835,988)
P(TP < 1) = 0.068868  (264,175/3,835,988)

ΔT (한국어 token − 영어 token, 이 실행에서 신규 계산):
  평균 6.9520 · 5% -1 · 25% 2 · 중앙 5 · 75% 10 · 95% 20
  P(ΔT>0) 0.879850 · P(ΔT=0) 0.051282 · P(ΔT<0) 0.068868
  교차 확인: P(ΔT>0) == P(TP>1) → True

[CARD01]
  CAN_SAY    : 이 병렬 말뭉치에서 o200k_base tokenizer 기준 한국어의 중앙 token premium은 33.3%였고, 문장쌍의 87.99%에서 한국어 token 수가 더 많았다.
  CANNOT_SAY : 한국어가 본질적으로 AI에 33% 비효율적이다 / 이 수치가 다른 tokenizer·다른 말뭉치·실제 서비스 비용에 그대로 적용된다 / token premium이 추론 성능 저하를 뜻한다.


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16.2, 4.8))
_ltp = qdf("SELECT log_token_premium FROM " + A)["log_token_premium"].to_numpy(np.float64)
_lo, _hi = float(np.quantile(_ltp, 0.0005)), float(np.quantile(_ltp, 0.9995))
axes[0].hist(_ltp, bins=360, range=(_lo, _hi), color="#3a7ca5", edgecolor="none")
axes[0].axvline(0.0, color="#0b0b0b", lw=1.3, ls="--", label="TP = 1 (프리미엄 없음)")
axes[0].axvline(C1["median_logTP"], color="#c1272d", lw=2.0,
                label=f"중앙값 {C1['median_logTP']:.4f} → TP = 4/3 (+33.3%)")
axes[0].set_xlabel("log Tokenization Premium = ln(한국어 token 수 / 영어 token 수)")
axes[0].set_ylabel("문장쌍 수")
axes[0].set_title("(가) 주 프리미엄 분포\n감사 등급 AUDITED_CANONICAL")
axes[0].legend(loc="upper left")

_dtv = qdf("SELECT token_difference FROM " + A)["token_difference"].to_numpy(np.float64)
axes[1].hist(_dtv, bins=int(_dt["q95"] - _dt["q05"]) + 41, range=(_dt["q05"] - 20, _dt["q95"] + 20),
             color="#7f9c96", edgecolor="none")
axes[1].axvline(0.0, color="#c1272d", lw=1.5, ls="--", label="ΔT = 0")
axes[1].axvline(_dt["q50"], color="#16425b", lw=1.8, label=f"중앙값 {_dt['q50']:.0f} token")
axes[1].set_xlabel("ΔT = 한국어 token 수 − 영어 token 수")
axes[1].set_ylabel("문장쌍 수")
axes[1].set_title("(나) 절대 token 차이 ΔT\n길이 의존 보조 지표 (DESCRIPTIVE)")
axes[1].legend()

_lab = ["한국어가 더 많음\n(TP > 1)", "정확히 같음\n(TP = 1)", "영어가 더 많음\n(TP < 1)"]
_val = [C1["P_TP_gt_1"], C1["P_TP_eq_1"], C1["P_TP_lt_1"]]
_cnt = [_pos, _tie, _neg]
_bars = axes[2].bar(_lab, [v * 100 for v in _val], color=["#c1272d", "#b9c6cc", "#0f4c81"])
for _b, _v, _c in zip(_bars, _val, _cnt, strict=True):
    axes[2].text(_b.get_x() + _b.get_width() / 2, _v * 100 + 1.5,
                 f"{_v * 100:.2f}%\n({_c:,}쌍)", ha="center", fontsize=9.5)
axes[2].set_ylabel("문장쌍 비율 (%)"); axes[2].set_ylim(0, 100)
axes[2].set_title("(다) 방향별 비율\n감사 등급 AUDITED_CANONICAL")
fig.suptitle(f"EP01 · 주 Tokenization Premium — o200k_base, 의미 대응 KO‑EN 문장쌍 {N:,}개",
             fontsize=13)
fig.tight_layout()
save_fig(fig, "EP01_primary_premium", "주 Tokenization Premium",
         card="CARD01", status="AUDITED_CANONICAL")
del _ltp, _dtv

  saved EP01_primary_premium  sha 4304fb13be83…  article_ready=YES


## CARD 02 — 한국어 대 영어 token 수

주의: **평균끼리 비교하는 것은 주 estimand가 아니다.** 주 estimand는 문장쌍 내부의
짝지어진 비(paired ratio)다. 두 언어의 주변 요약은 맥락 제공용으로만 싣는다.

In [7]:
C2 = {}
for _side, _ko in (("ko", "한국어"), ("en", "영어")):
    C2[_side] = quantiles(f"{_side}_token_count")
    for _m, _lab in (("mean", "평균"), ("q05", "5분위"), ("q25", "25분위"),
                     ("q50", "중앙값"), ("q75", "75분위"), ("q95", "95분위")):
        stat("CARD02", f"{_ko} token 수 {_lab}", f"{_side}_token_{_m}", C2[_side][_m],
             "DESCRIPTIVE_PERSISTED", source_artifact="D-04 TOKEN_O200K_BASE_v001",
             source_sha=D04_SHA, n=N)
C2["paired_ratio"] = quantiles("token_premium")
C2["paired_difference"] = C1["delta_T"]
C2["marginal_mean_ratio_NOT_the_estimand"] = C2["ko"]["mean"] / C2["en"]["mean"]

show(pd.DataFrame([
    {"언어": "한국어", **{k: C2["ko"][k] for k in ("mean", "q05", "q25", "q50", "q75", "q95")}},
    {"언어": "영어", **{k: C2["en"][k] for k in ("mean", "q05", "q25", "q50", "q75", "q95")}},
]).rename(columns={"mean": "평균", "q05": "5%", "q25": "25%", "q50": "중앙값",
                   "q75": "75%", "q95": "95%"}))
print(f"\n짝지어진 비 (token_premium) 중앙값 {C2['paired_ratio']['q50']:.6f} · "
      f"평균 {C2['paired_ratio']['mean']:.6f}")
print(f"짝지어진 차 (ΔT) 중앙값 {C2['paired_difference']['q50']:.0f} · "
      f"평균 {C2['paired_difference']['mean']:.4f}")
print(f"\n[주의] 주변 평균끼리의 비 = {C2['marginal_mean_ratio_NOT_the_estimand']:.6f} — "
      "이것은 주 estimand가 아니며 기사에 프리미엄으로 인용하면 안 된다.")

claim("C-04", "이 표본에서 한국어 문장은 중앙값 21개, 영어 문장은 중앙값 16개의 token으로 쪼개졌다.",
      "Median KO token count = 21, median EN token count = 16 (marginal summaries only).",
      "DESCRIPTIVE_PERSISTED", n=N,
      estimate=f"KO {C2['ko']['q50']:.0f} / EN {C2['en']['q50']:.0f}",
      ci_or_range=f"KO 5–95% [{C2['ko']['q05']:.0f}, {C2['ko']['q95']:.0f}] · "
                  f"EN 5–95% [{C2['en']['q05']:.0f}, {C2['en']['q95']:.0f}]",
      numerator="—", denominator=f"{N:,}",
      source_artifact="D-04 TOKEN_O200K_BASE_v001", source_SHA=D04_SHA, audit_SHA="",
      allowed_wording="두 언어의 주변 분포 요약",
      forbidden_wording="21/16 = 1.31 을 프리미엄으로 인용 (주변 평균·중앙값의 비는 주 estimand가 아니다)")

say("CARD02",
    "이 표본에서 한국어 문장은 중앙값 21 token, 영어 문장은 중앙값 16 token으로 쪼개졌다.",
    "두 중앙값의 비(21/16)를 token premium이라고 부르는 것 — 프리미엄은 문장쌍 내부에서 "
    "짝지어 계산한 비의 중앙값이지 주변 요약의 비가 아니다.")

 언어      평균  5%  25%  중앙값  75%  95%
한국어 27.1226   8   14   21   39   62
 영어 20.1706   6   10   16   28   46

짝지어진 비 (token_premium) 중앙값 1.333333 · 평균 1.362988
짝지어진 차 (ΔT) 중앙값 5 · 평균 6.9520

[주의] 주변 평균끼리의 비 = 1.344662 — 이것은 주 estimand가 아니며 기사에 프리미엄으로 인용하면 안 된다.

[CARD02]
  CAN_SAY    : 이 표본에서 한국어 문장은 중앙값 21 token, 영어 문장은 중앙값 16 token으로 쪼개졌다.
  CANNOT_SAY : 두 중앙값의 비(21/16)를 token premium이라고 부르는 것 — 프리미엄은 문장쌍 내부에서 짝지어 계산한 비의 중앙값이지 주변 요약의 비가 아니다.


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16.2, 4.8))
_tk = qdf(f"SELECT ko_token_count, en_token_count, token_premium FROM {A}")
_bins = np.logspace(0, np.log10(max(_tk["ko_token_count"].max(), _tk["en_token_count"].max())), 90)
axes[0].hist(_tk["ko_token_count"], bins=_bins, histtype="step", lw=2.0, color="#c1272d",
             label=f"한국어 (중앙값 {C2['ko']['q50']:.0f})")
axes[0].hist(_tk["en_token_count"], bins=_bins, histtype="step", lw=2.0, color="#0f4c81",
             label=f"영어 (중앙값 {C2['en']['q50']:.0f})")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("문장당 o200k_base token 수 (상용로그 눈금)")
axes[0].set_ylabel("문장 수 (로그 눈금)")
axes[0].set_title("(가) 언어별 token 수 주변 분포\n주 estimand가 아님"); axes[0].legend()

_x = np.arange(6); _w = 0.38
_labels = ["평균", "5%", "25%", "중앙값", "75%", "95%"]
_keys = ["mean", "q05", "q25", "q50", "q75", "q95"]
axes[1].bar(_x - _w / 2, [C2["ko"][k] for k in _keys], _w, color="#c1272d", label="한국어")
axes[1].bar(_x + _w / 2, [C2["en"][k] for k in _keys], _w, color="#0f4c81", label="영어")
for _i, _k in enumerate(_keys):
    axes[1].text(_i - _w / 2, C2["ko"][_k], f"{C2['ko'][_k]:.0f}", ha="center", va="bottom", fontsize=8)
    axes[1].text(_i + _w / 2, C2["en"][_k], f"{C2['en'][_k]:.0f}", ha="center", va="bottom", fontsize=8)
axes[1].set_xticks(_x, _labels); axes[1].set_ylabel("token 수")
axes[1].set_title("(나) 요약 통계 대조"); axes[1].legend()

_tp = _tk["token_premium"].to_numpy(np.float64)
axes[2].hist(_tp, bins=300, range=(0, 3.0), color="#8d6a9f", edgecolor="none")
axes[2].axvline(1.0, color="#0b0b0b", lw=1.3, ls="--", label="TP = 1")
axes[2].axvline(C2["paired_ratio"]["q50"], color="#c1272d", lw=2.0,
                label=f"짝지어진 비 중앙값 {C2['paired_ratio']['q50']:.4f}")
axes[2].axvline(C2["marginal_mean_ratio_NOT_the_estimand"], color="#e8a33d", lw=1.8, ls=":",
                label=f"주변 평균의 비 {C2['marginal_mean_ratio_NOT_the_estimand']:.4f}\n(estimand 아님)")
axes[2].set_xlabel("Tokenization Premium (문장쌍 내부 비)")
axes[2].set_ylabel("문장쌍 수")
axes[2].set_title("(다) 짝지어진 비 대 주변 평균의 비"); axes[2].legend(fontsize=8)
fig.suptitle(f"EP02 · 한국어 대 영어 token 수 (N = {N:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "EP02_token_counts", "한국어 대 영어 token 수",
         card="CARD02", status="DESCRIPTIVE_PERSISTED")
del _tk, _tp

  saved EP02_token_counts  sha 1539c834d0dd…  article_ready=NO


## CARD 03 — 정확 분해 (EXACT DECOMPOSITION)

항등식 `log TP = log CR + log BDR + log CP` 는 **G5에서 전수 감사**되었다 (오차 8.88e-16).
성분별 분포는 여기서 새로 계산한다.

**경고**: 성분들의 **중앙값을 곱해서 중앙 TP라고 부르면 안 된다.**
항등식은 문장쌍마다(pairwise) 성립하지, 주변 중앙값끼리 성립하지 않는다.

In [9]:
C3 = {"identity": "log TP = log CR + log BDR + log CP",
      "identity_status": "AUDITED_CANONICAL (G5 전수 검증, max |err| 8.88e-16)"}
_comp = {"log CR": "log_code_point_ratio", "log BDR": "log_byte_density_ratio",
         "log CP": "log_compression_penalty", "log TP": "log_token_premium"}
_rows = []
for _nm, _col in _comp.items():
    _s = quantiles(_col)
    _sg = q(f"""SELECT sum(CASE WHEN {_col} > 0 THEN 1 ELSE 0 END),
                       sum(CASE WHEN {_col} < 0 THEN 1 ELSE 0 END) FROM {A}""")
    C3[_nm] = {**_s, "share_gt_0": _sg[0] / N, "share_lt_0": _sg[1] / N,
               "exp_mean": float(np.exp(_s["mean"])), "exp_median": float(np.exp(_s["q50"]))}
    _rows.append({"성분": _nm, "평균": _s["mean"], "중앙값": _s["q50"],
                  "5%": _s["q05"], "25%": _s["q25"], "75%": _s["q75"], "95%": _s["q95"],
                  ">0 비율": _sg[0] / N, "<0 비율": _sg[1] / N,
                  "exp(평균)": C3[_nm]["exp_mean"], "exp(중앙값)": C3[_nm]["exp_median"]})
    stat("CARD03", f"{_nm} 중앙값", f"{_col}_median", _s["q50"], "DESCRIPTIVE_PERSISTED",
         source_artifact="D-02 + D-04", source_sha=f"{D02_SHA[:12]}…/{D04_SHA[:12]}…", n=N)
show(pd.DataFrame(_rows))

# §5C — 원 비율 지표도 분위와 부호 비율을 함께 싣는다
for _nm2, _col2 in (("pair_codepoint_ratio", "pair_codepoint_ratio"),
                    ("pair_byte_ratio", "pair_byte_ratio"),
                    ("byte_density_ratio", "exp(log_byte_density_ratio)"),
                    ("compression_penalty", "exp(log_compression_penalty)")):
    _s2 = quantiles(_col2)
    _sg2 = q(f"""SELECT sum(CASE WHEN {_col2} > 1 THEN 1 ELSE 0 END),
                        sum(CASE WHEN {_col2} = 1 THEN 1 ELSE 0 END),
                        sum(CASE WHEN {_col2} < 1 THEN 1 ELSE 0 END) FROM {A}""")
    C3[_nm2] = {**_s2, "share_gt_1": _sg2[0] / N, "share_eq_1": _sg2[1] / N,
                "share_lt_1": _sg2[2] / N}
    stat("CARD03", f"{_nm2} 중앙값", f"{_nm2}_median", _s2["q50"], "DESCRIPTIVE_PERSISTED",
         source_artifact="D-02 + D-04", source_sha=D02_SHA, n=N)
print("\n원 비율 지표 (§5C) — 분위와 부호 비율:")
show(pd.DataFrame([{"지표": k, "평균": C3[k]["mean"], "5%": C3[k]["q05"], "25%": C3[k]["q25"],
                    "중앙값": C3[k]["q50"], "75%": C3[k]["q75"], "95%": C3[k]["q95"],
                    ">1 비율": C3[k]["share_gt_1"], "=1 비율": C3[k]["share_eq_1"],
                    "<1 비율": C3[k]["share_lt_1"]}
                   for k in ("pair_codepoint_ratio", "pair_byte_ratio",
                             "byte_density_ratio", "compression_penalty")]))

_id_err = q(f"""SELECT max(abs(log_token_premium
                 - (log_code_point_ratio + log_byte_density_ratio + log_compression_penalty)))
              FROM {A}""")[0]
C3["identity_max_abs_error_recomputed"] = float(_id_err)
print(f"\n항등식 전수 재확인 (이 실행): max |오차| = {_id_err:.3e}  → 성립")
stat("CARD03", "정확 분해 항등식 성립", "identity_holds", True, "AUDITED_CANONICAL",
     source_artifact="G5 adjudication + 이 실행 재확인", source_sha=D04_SHA,
     audit_sha=G5_AUDIT_SHA, n=N, note=f"이 실행 재계산 max |err| {_id_err:.3e}")

# ---- 중앙값을 곱하면 안 되는 이유를 수치로 보인다 --------------------------------
_sum_of_medians = C3["log CR"]["q50"] + C3["log BDR"]["q50"] + C3["log CP"]["q50"]
C3["WARNING_sum_of_component_medians"] = float(_sum_of_medians)
C3["actual_median_logTP"] = C3["log TP"]["q50"]
C3["median_gap"] = float(_sum_of_medians - C3["log TP"]["q50"])
print(f"\n[금지 사례 시연]")
print(f"  성분 중앙값의 합            = {_sum_of_medians:.10f}  → exp = {np.exp(_sum_of_medians):.6f}")
print(f"  실제 log TP 중앙값          = {C3['log TP']['q50']:.10f}  → exp = "
      f"{np.exp(C3['log TP']['q50']):.6f}")
print(f"  차이                        = {C3['median_gap']:.10f}")
print("  → 중앙값은 합/곱에 대해 보존되지 않는다. 성분 중앙값의 곱을 '중앙 TP'로 부르면 틀린다.")

# ---- 폭포식(waterfall) 근거표: 평균 기준 ------------------------------------------
_wf = pd.DataFrame([
    {"단계": "① 한국어가 더 적은 문자 수", "성분": "log CR", "평균 기여": C3["log CR"]["mean"],
     "배수 환산 exp(평균)": C3["log CR"]["exp_mean"], "방향": "프리미엄을 낮춤"},
    {"단계": "② 한글의 높은 UTF-8 byte 밀도", "성분": "log BDR", "평균 기여": C3["log BDR"]["mean"],
     "배수 환산 exp(평균)": C3["log BDR"]["exp_mean"], "방향": "프리미엄을 높임"},
    {"단계": "③ 남는 tokenizer 압축 penalty", "성분": "log CP", "평균 기여": C3["log CP"]["mean"],
     "배수 환산 exp(평균)": C3["log CP"]["exp_mean"], "방향": "프리미엄을 높임"},
    {"단계": "= 관측된 프리미엄", "성분": "log TP", "평균 기여": C3["log TP"]["mean"],
     "배수 환산 exp(평균)": C3["log TP"]["exp_mean"], "방향": "합계"},
])
C3["waterfall_mean"] = _wf.to_dict(orient="records")
print("\n폭포식 근거표 (평균 기준, 세 단계의 합이 관측 프리미엄):")
show(_wf)

claim("C-05", "한국어의 token 프리미엄은 '글자가 많아서'가 아니다 — 글자 수는 오히려 적다.",
      "log CodePointRatio 중앙값 = -0.7605 (99.67%가 음수); 프리미엄은 byte 밀도와 압축 penalty "
      "항에서 나온다. 이는 대수적 항등식의 구성이지 인과 분해가 아니다.",
      "DESCRIPTIVE_PERSISTED", n=N, estimate="median log CR = -0.7605",
      ci_or_range="5–95% [-1.0986, -0.3868]", numerator="—", denominator=f"{N:,}",
      source_artifact="D-02 + D-04", source_SHA=D02_SHA, audit_SHA="",
      allowed_wording="문자 수 기준으로는 한국어가 더 짧다 (항등식의 세 항 중 하나)",
      forbidden_wording="문자 수가 프리미엄을 X% 설명한다 / 문자 수 때문에 프리미엄이 생긴다")
claim("C-06", "정확 분해 항등식은 383만 문장쌍 전수에서 성립한다.",
      "log TP = log CR + log BDR + log CP, max |error| = 8.88e-16 over N = 3,835,988 "
      "(G5 감사 + 이 실행 재확인).", "AUDITED_CANONICAL", n=N,
      estimate="max |error| 8.88e-16", ci_or_range="float64 반올림 한계", numerator="—",
      denominator=f"{N:,}", source_artifact="G5 adjudication", source_SHA=D04_SHA,
      audit_SHA=G5_AUDIT_SHA,
      allowed_wording="프리미엄은 세 요소로 정확히 분해된다 (수학적 항등식)",
      forbidden_wording="세 요소가 프리미엄의 원인을 각각 X%씩 차지한다")

say("CARD03",
    "관측된 token 프리미엄은 '문자 수 비 × UTF-8 byte 밀도 비 × tokenizer 압축 비'로 "
    "오차 없이 정확히 분해되며, 한국어는 문자 수에서는 오히려 짧다.",
    "세 요소 각각이 프리미엄의 원인을 몇 %씩 만든다 / 성분 중앙값을 곱해서 중앙 프리미엄을 "
    "구할 수 있다 / UTF-8이 3 byte라서 token이 3배가 된다.")

     성분        평균       중앙값         5%       25%       75%       95%      >0 비율       <0 비율  exp(평균)  exp(중앙값)
 log CR -0.753588 -0.760451   -1.09861 -0.895384 -0.619039 -0.386773 0.00193275    0.996691 0.470675  0.467456
log BDR  0.878476  0.896088   0.757096  0.856051  0.916291  0.944462   0.999653 5.47447e-06  2.40723      2.45
 log CP  0.160289  0.175154  -0.167054 0.0415764  0.290122  0.440663    0.80536     0.18844  1.17385   1.19143
 log TP  0.285177  0.287682 -0.0870114  0.154151  0.427444  0.635989    0.87985   0.0688675     1.33   1.33333



원 비율 지표 (§5C) — 분위와 부호 비율:
                  지표       평균       5%      25%      중앙값      75%      95%      >1 비율       =1 비율       <1 비율
pair_codepoint_ratio 0.482697 0.333333 0.408451 0.467456 0.538462 0.679245 0.00193275  0.00137592    0.996691
     pair_byte_ratio  1.16171 0.803714 0.984848  1.12676      1.3  1.63333   0.713563   0.0218948    0.264542
  byte_density_ratio  2.41221  2.13208  2.35385     2.45      2.5  2.57143   0.999653 0.000341503 5.47447e-06
 compression_penalty  1.19427 0.846154  1.04245  1.19143  1.33659  1.55374    0.80536  0.00620049     0.18844

항등식 전수 재확인 (이 실행): max |오차| = 8.882e-16  → 성립

[금지 사례 시연]
  성분 중앙값의 합            = 0.3107914455  → exp = 1.364505
  실제 log TP 중앙값          = 0.2876820725  → exp = 1.333333
  차이                        = 0.0231093730
  → 중앙값은 합/곱에 대해 보존되지 않는다. 성분 중앙값의 곱을 '중앙 TP'로 부르면 틀린다.

폭포식 근거표 (평균 기준, 세 단계의 합이 관측 프리미엄):
                       단계      성분     평균 기여  배수 환산 exp(평균)       방향
         ① 한국어가 더 적은 문자 수  log CR -0.753588   

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16.4, 5.0))
_dc = qdf(f"""SELECT log_code_point_ratio, log_byte_density_ratio,
                     log_compression_penalty, log_token_premium FROM {A}""")
_cols = {"log CR": ("#c1272d", "log CodePointRatio (문자 수 비)"),
         "log BDR": ("#e8a33d", "log ByteDensityRatio (byte 밀도 비)"),
         "log CP": ("#2f6690", "log CompressionPenalty (압축 penalty)"),
         "log TP": ("#3f3f3f", "log TokenizationPremium (합계)")}
_arr = {"log CR": _dc["log_code_point_ratio"].to_numpy(np.float64),
        "log BDR": _dc["log_byte_density_ratio"].to_numpy(np.float64),
        "log CP": _dc["log_compression_penalty"].to_numpy(np.float64),
        "log TP": _dc["log_token_premium"].to_numpy(np.float64)}
_lo = min(float(np.quantile(v, 0.001)) for v in _arr.values())
_hi = max(float(np.quantile(v, 0.999)) for v in _arr.values())
for _k in ("log TP", "log CR", "log BDR", "log CP"):
    axes[0].hist(_arr[_k], bins=320, range=(_lo, _hi), histtype="step", lw=1.9,
                 color=_cols[_k][0], label=_cols[_k][1])
axes[0].axvline(0.0, color="#0b0b0b", lw=1.0, ls=":")
axes[0].set_xlabel("로그 비율 값"); axes[0].set_ylabel("문장쌍 수")
axes[0].set_title("(가) 세 성분과 합계의 전수 분포"); axes[0].legend(fontsize=8, loc="upper left")

# 폭포 그래프 (평균 기준)
_steps = ["① 문자 수 비\n(log CR)", "② byte 밀도 비\n(log BDR)", "③ 압축 penalty\n(log CP)",
          "= 관측 프리미엄\n(log TP)"]
_vals = [C3["log CR"]["mean"], C3["log BDR"]["mean"], C3["log CP"]["mean"]]
_cum = np.concatenate([[0.0], np.cumsum(_vals)])
for _i, (_v, _c) in enumerate(zip(_vals, ["#c1272d", "#e8a33d", "#2f6690"], strict=True)):
    axes[1].bar(_i, _v, bottom=_cum[_i], color=_c, edgecolor="#333333", lw=0.7)
    axes[1].text(_i, _cum[_i] + _v / 2, f"{_v:+.4f}", ha="center", va="center",
                 fontsize=9.5, color="white", fontweight="bold")
    if _i < 2:
        axes[1].plot([_i + 0.4, _i + 0.6], [_cum[_i + 1]] * 2, color="#888888", lw=1.0, ls="--")
axes[1].bar(3, C3["log TP"]["mean"], color="#3f3f3f", edgecolor="#333333", lw=0.7)
axes[1].text(3, C3["log TP"]["mean"] / 2, f"{C3['log TP']['mean']:+.4f}", ha="center",
             va="center", fontsize=9.5, color="white", fontweight="bold")
axes[1].axhline(0.0, color="#0b0b0b", lw=1.0)
axes[1].set_xticks(range(4), _steps, fontsize=8.5)
axes[1].set_ylabel("로그 비율 값 (평균)")
axes[1].set_title("(나) 폭포식 분해 — 세 단계의 합이 관측 프리미엄\n"
                  "대수적 항등식이며 인과 분해가 아니다")

_tbl = [["", "exp(평균)", "exp(중앙값)"]] + [
    [k, f"{C3[k]['exp_mean']:.4f}", f"{C3[k]['exp_median']:.4f}"]
    for k in ("log CR", "log BDR", "log CP", "log TP")]
axes[2].axis("off")
_t = axes[2].table(cellText=_tbl[1:], colLabels=_tbl[0], loc="upper center", cellLoc="center")
_t.auto_set_font_size(False); _t.set_fontsize(10); _t.scale(1.0, 1.9)
axes[2].set_title("(다) 배수 환산\n주의: 성분 중앙값의 곱 ≠ 중앙 프리미엄", pad=18)
axes[2].text(0.5, 0.30,
             f"성분 중앙값의 합 {C3['WARNING_sum_of_component_medians']:.6f}\n"
             f"→ exp = {np.exp(C3['WARNING_sum_of_component_medians']):.6f}\n\n"
             f"실제 log TP 중앙값 {C3['log TP']['q50']:.6f}\n"
             f"→ 실제 중앙 TP = {np.exp(C3['log TP']['q50']):.6f}\n\n"
             f"차이 {C3['median_gap']:+.6f} — 중앙값은 합에 대해 보존되지 않는다",
             transform=axes[2].transAxes, ha="center", va="top", fontsize=9,
             bbox={"facecolor": "#fdf0f0", "edgecolor": "#c1272d", "boxstyle": "round,pad=0.6"})
fig.suptitle(f"EP03 · 정확 분해 log TP = log CR + log BDR + log CP (전수 N = {N:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "EP03_decomposition", "정확 분해", card="CARD03", status="DESCRIPTIVE_PERSISTED")
del _dc, _arr

  saved EP03_decomposition  sha d806a9455a6f…  article_ready=NO


## CARD 04 — 표현 역전 (REPRESENTATION REVERSAL)

**세 정의를 절대 같은 이름으로 부르지 않는다.** 각각 다른 것을 센다.

| 정의 | 조건 |
|---|---|
| A | 한국어 code point < 영어 code point |
| B | A **그리고** 한국어 UTF-8 byte > 영어 UTF-8 byte |
| C | A **그리고** 한국어 token 수 > 영어 token 수 |

부동소수점 경계 문제를 피하기 위해 **정수 열 직접 비교**로 계산한다.

In [11]:
_defs = {
    "A_codepoint_deficit": ("한국어 문자 수가 더 적음",
                            "ko_codepoint_count < en_codepoint_count"),
    "B_codepoint_deficit_AND_byte_surplus": ("문자는 적은데 byte는 더 많음 (표현층 내부 역전)",
                                             "ko_codepoint_count < en_codepoint_count "
                                             "AND ko_utf8_bytes > en_utf8_bytes"),
    "C_codepoint_deficit_AND_token_surplus": ("문자는 적은데 token은 더 많음 (표현→token 교차 역전)",
                                              "ko_codepoint_count < en_codepoint_count "
                                              "AND ko_token_count > en_token_count"),
}
C4 = {"denominator": N, "note": "정수 열 직접 비교 — 부동소수점 경계 배제"}
_rows = []
for _k, (_ko, _w) in _defs.items():
    _n = int(q(f"SELECT count(*) FROM {A} WHERE {_w}")[0])
    C4[_k] = {"label_ko": _ko, "predicate": _w, "count": _n, "share": _n / N,
              "numerator": _n, "denominator": N}
    _rows.append({"정의": _k[0], "설명": _ko, "n": _n, "share": _n / N, "조건": _w})
    stat("CARD04", f"표현 역전 {_k[0]} — {_ko}", _k, _n / N, "DESCRIPTIVE_PERSISTED",
         source_artifact="D-02 + D-04", source_sha=D02_SHA, n=N,
         numerator=f"{_n:,}", denominator=f"{N:,}")
show(pd.DataFrame(_rows).assign(share=lambda d: (d["share"] * 100).round(4)))

_ct = qdf(f"""SELECT
    CASE WHEN ko_utf8_bytes > en_utf8_bytes THEN 'byte 초과 O' ELSE 'byte 초과 X' END AS b,
    CASE WHEN ko_token_count > en_token_count THEN 'token 초과 O' ELSE 'token 초과 X' END AS t,
    count(*) AS n FROM {A}
  WHERE ko_codepoint_count < en_codepoint_count GROUP BY 1,2 ORDER BY 1,2""")
_ct["share_of_A"] = _ct["n"] / _ct["n"].sum()
C4["crosstab_within_A"] = json.loads(_ct.to_json(orient="records"))
print("\n정의 A 부분집합 내부 교차표 (B와 C는 서로 포함 관계가 아니다):")
show(_ct.assign(share_of_A=lambda d: (d["share_of_A"] * 100).round(4)))

claim("C-07", "한국어 문장은 글자 수로는 영어보다 짧지만, 실제 token 수는 더 많은 경우가 대부분이다.",
      "정의 C (codepoint deficit AND token surplus) = 3,363,717 / 3,835,988. 정의 B "
      "(codepoint deficit AND byte surplus) = 2,725,550. 두 집합은 서로 포함 관계가 아니다.",
      "DESCRIPTIVE_PERSISTED", n=N,
      estimate=f"C = {C4['C_codepoint_deficit_AND_token_surplus']['share'] * 100:.4f}%",
      ci_or_range=f"A = {C4['A_codepoint_deficit']['share'] * 100:.4f}% · "
                  f"B = {C4['B_codepoint_deficit_AND_byte_surplus']['share'] * 100:.4f}%",
      numerator=f"{C4['C_codepoint_deficit_AND_token_surplus']['count']:,}",
      denominator=f"{N:,}", source_artifact="D-02 + D-04", source_SHA=D02_SHA, audit_SHA="",
      allowed_wording="세 정의를 각각 이름 붙여 구분해 인용",
      forbidden_wording="세 수치를 모두 '표현 역전'이라는 한 이름으로 섞어 쓰기")

say("CARD04",
    "한국어는 문자 수로는 영어보다 짧은 경우가 99.67%인데, 그중에서도 token 수는 오히려 "
    "더 많은 경우가 전체의 87.69%다.",
    "99.67% · 71.05% · 87.69% 를 모두 같은 '역전' 수치로 섞어 인용하기 — 세 값은 각각 "
    "다른 조건을 센다.")

정의                                   설명       n   share                                                                          조건
 A                       한국어 문자 수가 더 적음 3823296 99.6691                                     ko_codepoint_count < en_codepoint_count
 B       문자는 적은데 byte는 더 많음 (표현층 내부 역전) 2725550 71.0521   ko_codepoint_count < en_codepoint_count AND ko_utf8_bytes > en_utf8_bytes
 C 문자는 적은데 token은 더 많음 (표현→token 교차 역전) 3363717 87.6884 ko_codepoint_count < en_codepoint_count AND ko_token_count > en_token_count



정의 A 부분집합 내부 교차표 (B와 C는 서로 포함 관계가 아니다):
        b          t       n  share_of_A
byte 초과 O token 초과 O 2583309     67.5676
byte 초과 O token 초과 X  142241      3.7204
byte 초과 X token 초과 O  780408     20.4119
byte 초과 X token 초과 X  317338      8.3001

[CARD04]
  CAN_SAY    : 한국어는 문자 수로는 영어보다 짧은 경우가 99.67%인데, 그중에서도 token 수는 오히려 더 많은 경우가 전체의 87.69%다.
  CANNOT_SAY : 99.67% · 71.05% · 87.69% 를 모두 같은 '역전' 수치로 섞어 인용하기 — 세 값은 각각 다른 조건을 센다.


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.0))
_labels = ["A. 문자 수만\n(조건 1개)", "B. 문자↓ & byte↑\n(표현층 내부)", "C. 문자↓ & token↑\n(표현→token)"]
_shares = [C4[k]["share"] for k in _defs]
_counts = [C4[k]["count"] for k in _defs]
_b = axes[0].bar(_labels, [s * 100 for s in _shares], color=["#b9c6cc", "#e8a33d", "#c1272d"])
for _bb, _s, _c in zip(_b, _shares, _counts, strict=True):
    axes[0].text(_bb.get_x() + _bb.get_width() / 2, _s * 100 + 1.5,
                 f"{_s * 100:.2f}%\n({_c:,}쌍)", ha="center", fontsize=10)
axes[0].set_ylabel("전체 문장쌍 대비 비율 (%)"); axes[0].set_ylim(0, 112)
axes[0].set_title("(가) 세 정의는 서로 다른 것을 센다\n같은 이름으로 부르면 안 된다")

_p = _ct.pivot(index="b", columns="t", values="n").fillna(0).astype(np.int64)
_arr2 = _p.to_numpy(dtype=float)
_pcm = axes[1].imshow(_arr2, cmap="YlOrRd", aspect="auto")
axes[1].set_xticks(range(_p.shape[1]), _p.columns)
axes[1].set_yticks(range(_p.shape[0]), _p.index)
for _i in range(_p.shape[0]):
    for _j in range(_p.shape[1]):
        _v = int(_arr2[_i, _j])
        axes[1].text(_j, _i, f"{_v:,}\n({_v / _arr2.sum() * 100:.2f}%)", ha="center",
                     va="center", fontsize=10,
                     color="white" if _v > _arr2.max() / 2 else "#14213d")
axes[1].set_title("(나) 정의 A 안에서 B와 C의 교차\n두 집합은 어느 쪽도 다른 쪽을 포함하지 않는다")
axes[1].grid(False)
fig.colorbar(_pcm, ax=axes[1], pad=0.02).set_label("문장쌍 수", fontsize=8)
fig.suptitle(f"EP04 · 표현 역전 — 세 정의의 분리 (N = {N:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "EP04_representation_reversal", "표현 역전 세 정의",
         card="CARD04", status="DESCRIPTIVE_PERSISTED")

  saved EP04_representation_reversal  sha 62a9da19239d…  article_ready=NO


{'figure_id': 'EP04_representation_reversal',
 'title_ko': '표현 역전 세 정의',
 'card': 'CARD04',
 'status': 'DESCRIPTIVE_PERSISTED',
 'article_ready': 'NO',
 'png': 'outputs/figures/evidence_pack/EP04_representation_reversal.png',
 'svg': 'outputs/figures/evidence_pack/EP04_representation_reversal.svg',
 'png_sha256': '62a9da19239d6c0aaf10a79e79ac50442ef1475f24e76c91aa06036e74b50983',
 'svg_sha256': '5f392145bcb4555f3fe149e49a658f5faa41bb278ef06a5f4dec8dfe29ca410f'}

## CARD 05 — UTF-8 부담 대 tokenizer 압축

목적: **byte 부담과 압축 penalty가 서로 다른 축**임을 시각적으로 보인다.
확률적 독립을 주장하지 않는다 — 순위 상관과 사분위 교차표만 제시한다.

사분위 정의는 이 노트북에서 고정한다: 각 축의 표본 사분위수(Q1/Q2/Q3)로 4등분하고,
"높음"은 상위 사분위(Q4), "낮음"은 하위 사분위(Q1)를 뜻한다.

In [13]:
_bc = qdf(f"SELECT log_byte_density_ratio AS bdr, log_compression_penalty AS cp FROM {A}")
_rho = float(stats.spearmanr(_bc["bdr"], _bc["cp"]).statistic)
_pear = float(np.corrcoef(_bc["bdr"], _bc["cp"])[0, 1])
C5 = {"spearman_logBDR_logCP": _rho, "pearson_logBDR_logCP": _pear,
      "quartile_rule": "각 축 표본 사분위수로 4등분; Q1=하위 25%, Q4=상위 25%"}
_qb = np.quantile(_bc["bdr"], [0.25, 0.5, 0.75])
_qc = np.quantile(_bc["cp"], [0.25, 0.5, 0.75])
C5["bdr_quartile_cuts"] = [float(x) for x in _qb]
C5["cp_quartile_cuts"] = [float(x) for x in _qc]
_bq = np.digitize(_bc["bdr"], _qb) + 1
_cq = np.digitize(_bc["cp"], _qc) + 1
_tab = pd.crosstab(pd.Series(_bq, name="BDR 사분위"), pd.Series(_cq, name="CP 사분위"))
C5["quartile_crosstab"] = json.loads(_tab.to_json(orient="index"))
_corners = {
    "high_BDR_high_CP": int(_tab.loc[4, 4]), "high_BDR_low_CP": int(_tab.loc[4, 1]),
    "low_BDR_high_CP": int(_tab.loc[1, 4]), "low_BDR_low_CP": int(_tab.loc[1, 1])}
C5["corners"] = _corners
C5["independence_expectation_per_cell"] = N / 16
stat("CARD05", "Spearman ρ(log BDR, log CP)", "spearman_bdr_cp", _rho, "DESCRIPTIVE_PERSISTED",
     source_artifact="D-02 + D-04", source_sha=D04_SHA, n=N)
print(f"Spearman ρ(log BDR, log CP) = {_rho:.6f}   (Pearson r = {_pear:.6f})")
print(f"사분위 경계 — BDR {np.round(_qb, 6).tolist()} · CP {np.round(_qc, 6).tolist()}")
print("\n사분위 교차표 (행 = BDR 사분위, 열 = CP 사분위):")
show(_tab.reset_index())
print(f"\n네 모서리: 높BDR·높CP {_corners['high_BDR_high_CP']:,} · "
      f"높BDR·낮CP {_corners['high_BDR_low_CP']:,} · "
      f"낮BDR·높CP {_corners['low_BDR_high_CP']:,} · "
      f"낮BDR·낮CP {_corners['low_BDR_low_CP']:,}")
print(f"(참고: 두 축이 무관하면 각 칸 기대값은 {N / 16:,.0f} — 이는 기대값 대조일 뿐 "
      "독립성 검정이 아니다)")

claim("C-08", "UTF-8 표현 부담이 큰 문장쌍이라고 해서 tokenizer가 그만큼 더 불리하게 압축하지는 않는다.",
      f"Spearman rho(log BDR, log CP) = {_rho:.4f} over N = 3,835,988; 사분위 교차표의 "
      "네 모서리가 모두 상당한 질량을 갖는다. 확률적 독립을 주장하지 않는다.",
      "DESCRIPTIVE_PERSISTED", n=N, estimate=f"rho = {_rho:.4f}",
      ci_or_range="점추정 — 구간 미제시", numerator="—", denominator=f"{N:,}",
      source_artifact="D-02 + D-04", source_SHA=D04_SHA, audit_SHA="",
      allowed_wording="두 지표는 이 표본에서 거의 함께 움직이지 않는다 (서로 다른 축)",
      forbidden_wording="byte 부담과 압축 penalty는 통계적으로 독립이다 / 서로 무관함이 증명되었다")

say("CARD05",
    "UTF-8 byte 부담과 tokenizer 압축 penalty는 이 표본에서 거의 함께 움직이지 않는 별개의 "
    "축이며, 네 조합(높·높 / 높·낮 / 낮·높 / 낮·낮)이 모두 실제로 관측된다.",
    "두 지표가 통계적으로 독립이다 / 상관이 0에 가깝다는 것이 인과적 무관함을 뜻한다.")

Spearman ρ(log BDR, log CP) = -0.050652   (Pearson r = -0.011039)
사분위 경계 — BDR [0.856051, 0.896088, 0.916291] · CP [0.041576, 0.175154, 0.290122]

사분위 교차표 (행 = BDR 사분위, 열 = CP 사분위):
 BDR 사분위      1      2      3      4
       1 227729 231767 248823 249852
       2 218269 239400 252108 248857
       3 163551 178055 185679 187071
       4 349305 309917 272388 273217

네 모서리: 높BDR·높CP 273,217 · 높BDR·낮CP 349,305 · 낮BDR·높CP 249,852 · 낮BDR·낮CP 227,729
(참고: 두 축이 무관하면 각 칸 기대값은 239,749 — 이는 기대값 대조일 뿐 독립성 검정이 아니다)

[CARD05]
  CAN_SAY    : UTF-8 byte 부담과 tokenizer 압축 penalty는 이 표본에서 거의 함께 움직이지 않는 별개의 축이며, 네 조합(높·높 / 높·낮 / 낮·높 / 낮·낮)이 모두 실제로 관측된다.
  CANNOT_SAY : 두 지표가 통계적으로 독립이다 / 상관이 0에 가깝다는 것이 인과적 무관함을 뜻한다.


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.2))
_x = _bc["bdr"].to_numpy(np.float64); _y = _bc["cp"].to_numpy(np.float64)
_rng = [[float(np.quantile(_x, 0.0005)), float(np.quantile(_x, 0.9995))],
        [float(np.quantile(_y, 0.0005)), float(np.quantile(_y, 0.9995))]]
_H, _xe, _ye = np.histogram2d(_x, _y, bins=220, range=_rng)
_pcm = axes[0].pcolormesh(_xe, _ye, np.ma.masked_where(_H.T == 0, _H.T),
                          norm=LogNorm(vmin=1, vmax=_H.max()), cmap="magma_r",
                          shading="auto", rasterized=True)
for _v in _qb:
    axes[0].axvline(_v, color="#0f8a5f", lw=1.0, ls=":")
for _v in _qc:
    axes[0].axhline(_v, color="#0f8a5f", lw=1.0, ls=":")
axes[0].set_xlabel("log ByteDensityRatio — UTF-8 표현 부담")
axes[0].set_ylabel("log CompressionPenalty — tokenizer 압축 열위")
axes[0].set_title(f"(가) 두 축의 결합 밀도\nSpearman ρ = {_rho:.4f} (점선 = 사분위 경계)")
fig.colorbar(_pcm, ax=axes[0], pad=0.02).set_label("문장쌍 수 (로그 눈금)", fontsize=8)

_a2 = _tab.to_numpy(dtype=float)
_pcm2 = axes[1].imshow(_a2, cmap="YlGnBu", aspect="auto")
axes[1].set_xticks(range(4), [f"CP Q{i}" for i in range(1, 5)])
axes[1].set_yticks(range(4), [f"BDR Q{i}" for i in range(1, 5)])
for _i in range(4):
    for _j in range(4):
        axes[1].text(_j, _i, f"{int(_a2[_i, _j]):,}", ha="center", va="center", fontsize=9,
                     color="white" if _a2[_i, _j] > _a2.max() / 2 else "#14213d")
axes[1].set_title("(나) 사분위 교차표 — 네 모서리가 모두 채워져 있다\n"
                  "byte 부담과 압축 penalty는 별개의 축")
axes[1].grid(False)
fig.colorbar(_pcm2, ax=axes[1], pad=0.02).set_label("문장쌍 수", fontsize=8)
fig.suptitle(f"EP05 · UTF-8 부담 대 tokenizer 압축 penalty (N = {N:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "EP05_byte_vs_compression", "UTF-8 부담 대 압축 penalty",
         card="CARD05", status="DESCRIPTIVE_PERSISTED")
del _bc

  saved EP05_byte_vs_compression  sha 323fd53a4b4c…  article_ready=NO


## CARD 06 — regex chunk → 최종 token 구조

기사에 안전하게 쓸 수 있는 관찰: **한국어는 regex chunk가 더 적은데도 최종 subword token은
더 많다.** 이는 기술적 관찰이며 "regex가 프리미엄을 만든다"는 진술이 아니다.
(SSOT MN‑02: chunk feature는 최종 token 수의 기계적 중간 단계 — mechanism audit로 분리)

In [15]:
C6 = {}
_rows = []
for _side, _ko in (("ko", "한국어"), ("en", "영어")):
    for _metric, _mko in (("chunk_count", "regex chunk 수"), ("token_count", "최종 token 수"),
                          ("tokens_per_chunk", "chunk당 token 수"),
                          ("max_tokens_per_chunk", "chunk당 최대 token 수")):
        _col = f"{_side}_{_metric}"
        _s = quantiles(_col)
        C6[_col] = _s
        _rows.append({"언어": _ko, "지표": _mko, "평균": _s["mean"], "5%": _s["q05"],
                      "25%": _s["q25"], "중앙값": _s["q50"], "75%": _s["q75"], "95%": _s["q95"]})
        stat("CARD06", f"{_ko} {_mko} 중앙값", f"{_col}_median", _s["q50"],
             "DESCRIPTIVE_PERSISTED", source_artifact="D-05 + D-04",
             source_sha=D05_SHA, n=N)
show(pd.DataFrame(_rows))

_cmp = q(f"""SELECT
    sum(CASE WHEN ko_chunk_count < en_chunk_count THEN 1 ELSE 0 END),
    sum(CASE WHEN ko_chunk_count < en_chunk_count AND ko_token_count > en_token_count
             THEN 1 ELSE 0 END) FROM {A}""")
C6["ko_fewer_chunks"] = {"n": int(_cmp[0]), "share": _cmp[0] / N}
C6["ko_fewer_chunks_but_more_tokens"] = {"n": int(_cmp[1]), "share": _cmp[1] / N}
print(f"\n한국어 chunk 수가 더 적은 문장쌍            : {_cmp[0]:,} ({_cmp[0] / N * 100:.4f}%)")
print(f"그런데 최종 token 수는 더 많은 문장쌍        : {_cmp[1]:,} ({_cmp[1] / N * 100:.4f}%)")
stat("CARD06", "chunk는 적은데 token은 많은 문장쌍 비율", "ko_fewer_chunks_more_tokens",
     _cmp[1] / N, "DESCRIPTIVE_PERSISTED", source_artifact="D-05 + D-04", source_sha=D05_SHA,
     n=N, numerator=f"{int(_cmp[1]):,}", denominator=f"{N:,}")

# §5E — 짝지어진 차이도 함께 싣는다
_pairdiff = {}
for _m2, _ko2 in (("chunk_count", "regex chunk 수 차이 (한국어 − 영어)"),
                  ("tokens_per_chunk", "chunk당 token 수 차이"),
                  ("token_count", "최종 token 수 차이")):
    _s3 = quantiles(f"(ko_{_m2} - en_{_m2})")
    _pairdiff[_m2] = _s3
    stat("CARD06", f"{_ko2} 중앙값", f"paired_diff_{_m2}_median", _s3["q50"],
         "DESCRIPTIVE_PERSISTED", source_artifact="D-05 + D-04", source_sha=D05_SHA, n=N)
C6["paired_differences"] = _pairdiff
print("\n짝지어진 차이 (§5E):")
show(pd.DataFrame([{"지표": k, "평균": v["mean"], "5%": v["q05"], "25%": v["q25"],
                    "중앙값": v["q50"], "75%": v["q75"], "95%": v["q95"]}
                   for k, v in _pairdiff.items()]))

claim("C-09", "한국어는 tokenizer가 나누는 덩어리(regex chunk)는 오히려 더 적은데, "
              "최종 subword token은 더 많다.",
      f"KO median chunk = {C6['ko_chunk_count']['q50']:.0f}, EN = {C6['en_chunk_count']['q50']:.0f}; "
      f"KO median tokens-per-chunk = {C6['ko_tokens_per_chunk']['q50']:.1f}, "
      f"EN = {C6['en_tokens_per_chunk']['q50']:.1f}. 순수 기술 관찰.",
      "DESCRIPTIVE_PERSISTED", n=N,
      estimate=f"chunk 적고 token 많은 쌍 {_cmp[1] / N * 100:.2f}%",
      ci_or_range="—", numerator=f"{int(_cmp[1]):,}", denominator=f"{N:,}",
      source_artifact="D-05 + D-04", source_SHA=D05_SHA, audit_SHA="",
      allowed_wording="덩어리 하나가 더 많은 조각으로 쪼개진다 (기술적 관찰)",
      forbidden_wording="regex chunking이 프리미엄을 만든다 / regex 규칙이 한국어에 불리하게 설계됐다")

say("CARD06",
    "한국어 문장은 tokenizer가 먼저 나누는 덩어리(regex chunk)가 영어보다 오히려 적은데도, "
    "덩어리 하나가 평균 2개 이상의 조각으로 쪼개지면서 최종 token 수는 더 많아진다.",
    "regex chunking이 프리미엄의 원인이다 / 이 단계만 고치면 프리미엄이 사라진다 — "
    "chunk 지표는 최종 token 수의 기계적 중간 단계이며 인과 진술의 근거가 아니다.")

 언어                지표      평균      5%  25%  중앙값     75%     95%
한국어     regex chunk 수 13.5611       4    7   11      19      31
한국어        최종 token 수 27.1226       8   14   21      39      62
한국어    chunk당 token 수 2.01862 1.48649 1.75    2    2.25 2.66667
한국어 chunk당 최대 token 수 3.98277       2    3    4       5       6
 영어     regex chunk 수 19.4989       6   10   15      27      44
 영어        최종 token 수 20.1706       6   10   16      28      46
 영어    chunk당 token 수 1.03649       1    1    1 1.04762 1.16667
 영어 chunk당 최대 token 수 1.43506       1    1    1       2       3

한국어 chunk 수가 더 적은 문장쌍            : 3,559,410 (92.7899%)
그런데 최종 token 수는 더 많은 문장쌍        : 3,107,277 (81.0033%)



짝지어진 차이 (§5E):
              지표       평균       5%      25%      중앙값  75%     95%
     chunk_count  -5.9378      -17       -8       -5   -2       0
tokens_per_chunk 0.982127 0.438889 0.735714 0.976923  1.2 1.61538
     token_count  6.95204       -1        2        5   10      20

[CARD06]
  CAN_SAY    : 한국어 문장은 tokenizer가 먼저 나누는 덩어리(regex chunk)가 영어보다 오히려 적은데도, 덩어리 하나가 평균 2개 이상의 조각으로 쪼개지면서 최종 token 수는 더 많아진다.
  CANNOT_SAY : regex chunking이 프리미엄의 원인이다 / 이 단계만 고치면 프리미엄이 사라진다 — chunk 지표는 최종 token 수의 기계적 중간 단계이며 인과 진술의 근거가 아니다.


In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16.2, 4.8))
_ck = qdf(f"""SELECT ko_chunk_count, en_chunk_count, ko_tokens_per_chunk,
                     en_tokens_per_chunk, ko_token_count, en_token_count FROM {A}""")
_x = np.arange(2); _w = 0.36
for _k, (_m, _lab, _c) in enumerate((("chunk_count", "regex chunk 수", "#e8a33d"),
                                     ("token_count", "최종 token 수", "#c1272d"))):
    axes[0].bar(_x[_k] - _w / 2, C6[f"ko_{_m}"]["q50"], _w, color=_c, label="한국어" if _k == 0 else None)
    axes[0].bar(_x[_k] + _w / 2, C6[f"en_{_m}"]["q50"], _w, color=_c, alpha=0.5, hatch="//",
                label="영어" if _k == 0 else None)
    axes[0].text(_x[_k] - _w / 2, C6[f"ko_{_m}"]["q50"], f"{C6[f'ko_{_m}']['q50']:.0f}",
                 ha="center", va="bottom", fontsize=10)
    axes[0].text(_x[_k] + _w / 2, C6[f"en_{_m}"]["q50"], f"{C6[f'en_{_m}']['q50']:.0f}",
                 ha="center", va="bottom", fontsize=10)
axes[0].set_xticks(_x, ["regex chunk 수", "최종 subword token 수"])
axes[0].set_ylabel("문장당 개수 (중앙값)")
axes[0].set_title("(가) 순서가 뒤집힌다\n덩어리는 한국어가 적고, 조각은 한국어가 많다")
axes[0].legend()

axes[1].hist(_ck["ko_tokens_per_chunk"], bins=200, range=(0, 6), histtype="step", lw=2.0,
             color="#c1272d", label=f"한국어 (중앙값 {C6['ko_tokens_per_chunk']['q50']:.1f})")
axes[1].hist(_ck["en_tokens_per_chunk"], bins=200, range=(0, 6), histtype="step", lw=2.0,
             color="#0f4c81", label=f"영어 (중앙값 {C6['en_tokens_per_chunk']['q50']:.1f})")
axes[1].set_yscale("log")
axes[1].set_xlabel("chunk 하나가 쪼개지는 token 수"); axes[1].set_ylabel("문장 수 (로그 눈금)")
axes[1].set_title("(나) 덩어리 하나가 몇 조각이 되는가"); axes[1].legend()

_lb = ["한국어 chunk 수가\n더 적음", "그중 token 수는\n오히려 더 많음"]
_vv = [C6["ko_fewer_chunks"]["share"], C6["ko_fewer_chunks_but_more_tokens"]["share"]]
_bb = axes[2].bar(_lb, [v * 100 for v in _vv], color=["#b9c6cc", "#c1272d"])
for _b2, _v in zip(_bb, _vv, strict=True):
    axes[2].text(_b2.get_x() + _b2.get_width() / 2, _v * 100 + 1.2,
                 f"{_v * 100:.2f}%", ha="center", fontsize=11)
axes[2].set_ylabel("전체 문장쌍 대비 비율 (%)"); axes[2].set_ylim(0, 100)
axes[2].set_title("(다) 기술적 관찰\n인과 진술이 아니다")
fig.suptitle(f"EP06 · regex chunk에서 최종 token까지 (o200k_base, N = {N:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "EP06_chunk_to_token", "regex chunk에서 최종 token까지",
         card="CARD06", status="DESCRIPTIVE_PERSISTED")
del _ck

  saved EP06_chunk_to_token  sha 0c82f66fdf36…  article_ready=NO


## CARD 07 — 세 분석 계층 (핵심 설명 인포그래픽)

**언어학적 형태소 ≠ regex chunk ≠ 최종 subword token.**
세 계층은 서로 다른 주체가 만들고, 서로 포함 관계가 없으며, 버전 민감도도 다르다
(SSOT §5).

In [17]:
_L = q(f"""SELECT avg(ko_eojeol_count), quantile_cont(ko_eojeol_count, 0.5),
                  avg(morpheme_count), quantile_cont(morpheme_count, 0.5),
                  avg(ko_chunk_count), quantile_cont(ko_chunk_count, 0.5),
                  avg(ko_token_count), quantile_cont(ko_token_count, 0.5),
                  avg(en_word_count), quantile_cont(en_word_count, 0.5),
                  avg(en_chunk_count), quantile_cont(en_chunk_count, 0.5),
                  avg(en_token_count), quantile_cont(en_token_count, 0.5) FROM {A}""")
LAYERS = pd.DataFrame([
    {"계층": "① 언어학적 형태소 분석", "단위": "어절 (eojeol)", "산출 주체": "Kiwi 분석기 (D-03)",
     "버전 민감도": "분석기·모델 의존", "한국어 평균": _L[0], "한국어 중앙값": _L[1],
     "영어 대응": "단어(공백 분리)", "영어 평균": _L[8], "영어 중앙값": _L[9],
     "연구상 역할": "외생적 설명 feature"},
    {"계층": "① 언어학적 형태소 분석", "단위": "형태소 (morpheme)", "산출 주체": "Kiwi 분석기 (D-03)",
     "버전 민감도": "분석기·모델 의존", "한국어 평균": _L[2], "한국어 중앙값": _L[3],
     "영어 대응": "정의 없음", "영어 평균": np.nan, "영어 중앙값": np.nan,
     "연구상 역할": "외생적 설명 feature"},
    {"계층": "② o200k_base regex chunking", "단위": "regex chunk",
     "산출 주체": "tiktoken pat_str (D-05)", "버전 민감도": "tokenizer 구현 의존",
     "한국어 평균": _L[4], "한국어 중앙값": _L[5], "영어 대응": "regex chunk",
     "영어 평균": _L[10], "영어 중앙값": _L[11], "연구상 역할": "tokenizer mechanism audit"},
    {"계층": "③ 최종 subword tokenization", "단위": "o200k_base token",
     "산출 주체": "tiktoken BPE merge (D-04)", "버전 민감도": "매우 큼",
     "한국어 평균": _L[6], "한국어 중앙값": _L[7], "영어 대응": "o200k_base token",
     "영어 평균": _L[12], "영어 중앙값": _L[13], "연구상 역할": "결과변수 생성 과정"},
])
C7 = {"table": LAYERS.replace({np.nan: None}).to_dict(orient="records"),
      "ssot_ref": "KOEN-TP-RS-001 §5 · RR-01 · MN-02"}
show(LAYERS, "{:,.3f}")
for _, _r in LAYERS.iterrows():
    _unit = str(_r["단위"]).split(" ")[0]
    stat("CARD07", f"{_r['단위']} 한국어 중앙값", f"layer_{_unit}_ko_median",
         float(_r["한국어 중앙값"]), "DESCRIPTIVE_PERSISTED",
         source_artifact="D-02 + D-03 + D-04 + D-05", source_sha=D03_SHA, n=N)
print("\n※ SSOT §5: 언어학적 경계는 ①에서만 '중요'하고 ②·③에서는 '보장하지 않음'이다.")

claim("C-10", "형태소 분석 단위, tokenizer가 나누는 덩어리, 최종 token은 서로 다른 세 계층이다.",
      f"KO 중앙값: 어절 {_L[1]:.0f} → 형태소 {_L[3]:.0f} → regex chunk {_L[5]:.0f} → "
      f"subword token {_L[7]:.0f}. 단조 정렬조차 되지 않는다 (포함 관계 없음).",
      "DESCRIPTIVE_PERSISTED", n=N,
      estimate=f"{_L[1]:.0f} → {_L[3]:.0f} → {_L[5]:.0f} → {_L[7]:.0f}",
      ci_or_range="중앙값", numerator="—", denominator=f"{N:,}",
      source_artifact="D-02 + D-03 + D-04 + D-05", source_SHA=D03_SHA, audit_SHA="",
      allowed_wording="세 계층은 다른 단위를 만든다 (개수 비교)",
      forbidden_wording="형태소 경계가 token 경계와 일치한다 / 한 계층이 다른 계층을 결정한다")

say("CARD07",
    "같은 한국어 문장이 형태소 분석에서는 형태소 19개, tokenizer의 regex 단계에서는 덩어리 "
    "11개, 최종적으로는 token 21개가 된다 — 세 계층은 서로 포함 관계가 없는 다른 단위다.",
    "형태소가 많아서 token이 많다 / 형태소 경계와 token 경계가 대응한다 — 경계 일치 여부는 "
    "측정하지 않았고, 개수 비교는 대응 관계의 증거가 아니다.")

                         계층               단위                     산출 주체          버전 민감도  한국어 평균  한국어 중앙값            영어 대응  영어 평균  영어 중앙값                    연구상 역할
              ① 언어학적 형태소 분석      어절 (eojeol)           Kiwi 분석기 (D-03)       분석기·모델 의존  10.951    9.000        단어(공백 분리) 16.856  13.000            외생적 설명 feature
              ① 언어학적 형태소 분석   형태소 (morpheme)           Kiwi 분석기 (D-03)       분석기·모델 의존  24.153   19.000            정의 없음    NaN     NaN            외생적 설명 feature
② o200k_base regex chunking      regex chunk   tiktoken pat_str (D-05) tokenizer 구현 의존  13.561   11.000      regex chunk 19.499  15.000 tokenizer mechanism audit
  ③ 최종 subword tokenization o200k_base token tiktoken BPE merge (D-04)            매우 큼  27.123   21.000 o200k_base token 20.171  16.000                결과변수 생성 과정

※ SSOT §5: 언어학적 경계는 ①에서만 '중요'하고 ②·③에서는 '보장하지 않음'이다.

[CARD07]
  CAN_SAY    : 같은 한국어 문장이 형태소 분석에서는 형태소 19개, tokenizer의 regex 단계에서는 덩어리 11개, 최종적으로는 token 21개가 된다 — 세 계층은 서로 포함 관계가 없는 다른 단위다.


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(15.0, 5.2))
_ko_layers = [("어절\n①", _L[1], "#7f9c96"), ("형태소\n①", _L[3], "#4c8577"),
              ("regex chunk\n②", _L[5], "#e8a33d"), ("subword token\n③", _L[7], "#c1272d")]
_en_layers = [("단어\n①", _L[9], "#7f9c96"), ("—", np.nan, "#ffffff"),
              ("regex chunk\n②", _L[11], "#e8a33d"), ("subword token\n③", _L[13], "#0f4c81")]
for _ax, _ly, _nm in ((axes[0], _ko_layers, "한국어"), (axes[1], _en_layers, "영어")):
    _v = [x[1] for x in _ly]
    _ax.bar(range(len(_ly)), _v, color=[x[2] for x in _ly], edgecolor="#333333", lw=0.7)
    for _i, _vv in enumerate(_v):
        if np.isfinite(_vv):
            _ax.text(_i, _vv, f"{_vv:,.0f}", ha="center", va="bottom", fontsize=11)
    _ax.set_xticks(range(len(_ly)), [x[0] for x in _ly], fontsize=9)
    _ax.set_ylabel("문장당 단위 수 (중앙값)")
    _ax.set_ylim(0, max(v for v in _v if np.isfinite(v)) * 1.25)
    _ax.axvspan(-0.5, 1.5, color="#7f9c96", alpha=0.10)
    _ax.axvspan(1.5, 3.5, color="#e8a33d", alpha=0.10)
    _ax.text(0.5, _ax.get_ylim()[1] * 0.94, "① 언어학적 계층", ha="center", fontsize=9.5,
             color="#3f6b5f")
    _ax.text(2.5, _ax.get_ylim()[1] * 0.94, "②③ tokenizer 계층", ha="center", fontsize=9.5,
             color="#96631f")
    _ax.set_title(f"{_nm} — 계층마다 단위 수가 다르다")
fig.suptitle("EP07 · 세 분석 계층: 언어학적 형태소 ≠ regex chunking ≠ 최종 subword tokenization\n"
             "(SSOT §5 — 언어학적 경계는 ①에서만 보장된다)", fontsize=12.5)
fig.tight_layout()
save_fig(fig, "EP07_three_layer_boundary", "세 분석 계층 경계",
         card="CARD07", status="DESCRIPTIVE_PERSISTED")

  saved EP07_three_layer_boundary  sha d3905a332f13…  article_ready=NO


{'figure_id': 'EP07_three_layer_boundary',
 'title_ko': '세 분석 계층 경계',
 'card': 'CARD07',
 'status': 'DESCRIPTIVE_PERSISTED',
 'article_ready': 'NO',
 'png': 'outputs/figures/evidence_pack/EP07_three_layer_boundary.png',
 'svg': 'outputs/figures/evidence_pack/EP07_three_layer_boundary.svg',
 'png_sha256': 'd3905a332f1309c5cdcdd53a4ac67ec45a52a1e7cbc30165014e9df2691358d1',
 'svg_sha256': '20cb96e201c97d2b7f06f74be150b707562982d614069e1639c4154dcc59a064'}

## CARD 08 — 도메인 · 출처 · 방향 support

모든 층별 수치는 **기술적 층(DESCRIPTIVE STRATUM)** 으로만 표기한다.
**순수 source 효과나 순수 domain 효과는 이 자료에서 식별되지 않는다** (G5 ID‑03 · ID‑04).

In [19]:
C8 = {"label": "DESCRIPTIVE STRATUM — not a domain effect, not a source effect",
      "identifiability_refs": ["G5 ID-03", "G5 ID-04"]}
_cell = qdf(f"""SELECT source_domain_cell AS cell, count(*) AS n,
    quantile_cont(token_premium, 0.5) AS median_TP,
    quantile_cont(log_token_premium, 0.5) AS median_logTP,
    avg(CASE WHEN token_premium > 1 THEN 1.0 ELSE 0.0 END) AS P_TP_gt_1
  FROM {A} GROUP BY 1 ORDER BY 2 DESC""")
_cell["share"] = _cell["n"] / N
C8["cells"] = json.loads(_cell.to_json(orient="records"))
print("관측된 source_domain_cell 전체 (기술적 층):")
show(_cell[["cell", "n", "share", "median_TP", "median_logTP", "P_TP_gt_1"]])
for _, _r in _cell.iterrows():
    stat("CARD08", f"{_r['cell']} 중앙 TP", f"cell_{_r['cell']}_median_TP", float(_r["median_TP"]),
         "DESCRIPTIVE_PERSISTED", source_artifact="D-01 + D-04", source_sha=D04_SHA,
         n=int(_r["n"]), note="DESCRIPTIVE STRATUM — 순수 source/domain 효과 아님")

_sd = qdf(f"SELECT source, domain, count(*) n FROM {A} GROUP BY 1,2")
SD = _sd.pivot(index="source", columns="domain", values="n").reindex(
    columns=["dialogue", "general", "other", "technology"]).fillna(0).astype(np.int64)
_cd = qdf(f"SELECT source_domain_cell AS cell, translation_direction AS d, count(*) n "
          f"FROM {A} GROUP BY 1,2")
CD = _cd.pivot(index="cell", columns="d", values="n").reindex(
    columns=["KO_TO_EN", "EN_TO_KO", "UNKNOWN"]).fillna(0).astype(np.int64)
C8["source_x_domain"] = json.loads(SD.to_json(orient="index"))
C8["cell_x_direction"] = json.loads(CD.to_json(orient="index"))
C8["shared_domains"] = list(SD.columns[(SD > 0).all(axis=0)])
C8["n_026_EN_TO_KO"] = int(CD.loc[[i for i in CD.index if i.startswith("026")], "EN_TO_KO"].sum())
C8["n_direction_unknown"] = int(CD["UNKNOWN"].sum())
print("\n출처 × 도메인 support:"); show(SD.reset_index())
print(f"두 출처 모두에서 관측되는 도메인: {C8['shared_domains']}  (그 외는 한쪽 전용)")
print("\n출처-도메인 셀 × 번역 방향 support:"); show(CD.reset_index())
print(f"026 계열의 EN_TO_KO 총계 = {C8['n_026_EN_TO_KO']}")
print(f"방향 UNKNOWN 총계 = {C8['n_direction_unknown']:,} (제거하지 않는다)")
for _k, _v, _ko in (("shared_domain_count", len(C8["shared_domains"]), "두 출처 공유 도메인 수"),
                    ("n_026_EN_TO_KO", C8["n_026_EN_TO_KO"], "026 계열 EN_TO_KO 관측 수"),
                    ("n_direction_unknown", C8["n_direction_unknown"], "방향 UNKNOWN 수")):
    stat("CARD08", _ko, _k, _v, "AUDITED_CANONICAL",
         source_artifact="G5_IDENTIFIABILITY (재확인)", source_sha=D04_SHA,
         audit_sha=G5_AUDIT_SHA, n=N)

claim("C-11", "출처와 도메인이 거의 겹치지 않아, '어느 분야가 더 불리한가'를 이 자료로 답할 수 없다.",
      "source × domain 실현 셀 5/8; `other`만 두 출처에서 관측된다. source와 domain의 "
      "독립 주효과는 식별 불가 (G5 ID-03). 026 계열에는 EN_TO_KO 관측이 0건 (G5 ID-04).",
      "AUDITED_CANONICAL", n=N, estimate="실현 셀 5 / 8", ci_or_range="—",
      numerator="5", denominator="8", source_artifact="G5_IDENTIFIABILITY", source_SHA=D04_SHA,
      audit_SHA=G5_AUDIT_SHA,
      allowed_wording="관측 층별 기술 수치 / 식별 가능한 비교 범위",
      forbidden_wording="법률 도메인이 기술 도메인보다 프리미엄이 높다 / 출처 A가 출처 B보다 불리하다")

say("CARD08",
    "관측된 다섯 개 출처-도메인 층 모두에서 한국어 쪽 token이 더 많았고, 층별 중앙 프리미엄은 "
    "1.286배에서 1.414배 사이였다.",
    "특정 도메인이나 특정 출처가 프리미엄을 높인다 — 출처와 도메인이 거의 완전히 겹쳐 있어 "
    "두 효과를 분리할 수 없다.")

관측된 source_domain_cell 전체 (기술적 층):
          cell       n     share  median_TP  median_logTP  P_TP_gt_1
     025-other 1165510  0.303836    1.35714      0.305382   0.880291
     026-other  990120  0.258113    1.34091      0.293348   0.955121
   025-general  804291   0.20967    1.28571      0.251314    0.77372
  025-dialogue  516162  0.134558    1.28571      0.251314   0.821587
026-technology  359905 0.0938233    1.41379      0.346276   0.992081

출처 × 도메인 support:
source  dialogue  general   other  technology
   025    516162   804291 1165510           0
   026         0        0  990120      359905
두 출처 모두에서 관측되는 도메인: ['other']  (그 외는 한쪽 전용)

출처-도메인 셀 × 번역 방향 support:
          cell  KO_TO_EN  EN_TO_KO  UNKNOWN
  025-dialogue    247947    254955    13260
   025-general    354654    449606       31
     025-other    559527    568728    37255
     026-other    990119         0        1
026-technology    359905         0        0
026 계열의 EN_TO_KO 총계 = 0
방향 UNKNOWN 총계 = 50,547 (제거하지 않는다)



In [20]:
fig, axes = plt.subplots(1, 3, figsize=(17.0, 5.0))
_c = _cell.sort_values("median_TP")
_b = axes[0].barh(_c["cell"], _c["median_TP"],
                  color=["#c1272d" if s.startswith("025") else "#0f4c81" for s in _c["cell"]])
for _bb, _v, _n2 in zip(_b, _c["median_TP"], _c["n"], strict=True):
    axes[0].text(_v, _bb.get_y() + _bb.get_height() / 2, f"  {_v:.4f}배 (n={_n2:,})",
                 va="center", fontsize=8.5)
axes[0].axvline(1.0, color="#0b0b0b", lw=1.2, ls="--", label="프리미엄 없음")
axes[0].set_xlim(0.9, 1.62); axes[0].set_xlabel("중앙 Tokenization Premium (배)")
axes[0].set_title("(가) 기술적 층별 중앙 프리미엄\nDESCRIPTIVE STRATUM — 효과가 아니다")
axes[0].legend(loc="lower right")


def _support(ax, mat, title, xlab, ylab):
    arr = mat.to_numpy(dtype=float)
    cm = plt.get_cmap("YlGnBu").copy(); cm.set_bad("#f2c6c6")
    pc = ax.imshow(np.ma.masked_where(arr == 0, arr), cmap=cm, aspect="auto",
                   norm=LogNorm(vmin=max(arr[arr > 0].min(), 1), vmax=arr.max()))
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=18, ha="right")
    ax.set_yticks(range(mat.shape[0]), mat.index)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = int(arr[i, j])
            ax.text(j, i, "없음\n(0)" if v == 0 else f"{v:,}", ha="center", va="center",
                    fontsize=8, fontweight="bold" if v == 0 or 0 < v <= 100 else "normal",
                    color="#7a1f1f" if v == 0 else ("white" if v > arr.max() / 6 else "#14213d"))
    ax.set_title(title); ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(False)


_support(axes[1], SD, "(나) 출처 × 도메인 — 8칸 중 3칸이 비어 있다\n`other`만 두 출처 공통",
         "도메인", "출처")
_support(axes[2], CD, "(다) 층 × 번역 방향 — 026에 EN_TO_KO 0건\nUNKNOWN은 제거하지 않는다",
         "번역 방향", "출처-도메인 셀")
fig.suptitle(f"EP08 · 기술적 층과 관측 support — 순수 출처/도메인 효과는 식별되지 않는다 (N = {N:,})",
             fontsize=12.5)
fig.tight_layout()
save_fig(fig, "EP08_heterogeneity", "기술적 층과 관측 support",
         card="CARD08", status="DESCRIPTIVE_PERSISTED")

  saved EP08_heterogeneity  sha eed288149c78…  article_ready=NO


{'figure_id': 'EP08_heterogeneity',
 'title_ko': '기술적 층과 관측 support',
 'card': 'CARD08',
 'status': 'DESCRIPTIVE_PERSISTED',
 'article_ready': 'NO',
 'png': 'outputs/figures/evidence_pack/EP08_heterogeneity.png',
 'svg': 'outputs/figures/evidence_pack/EP08_heterogeneity.svg',
 'png_sha256': 'eed288149c78ccfa087e792eb2337770206292d573886bc918fae9d5095cb0f1',
 'svg_sha256': '36b60d149be8eb719bfdbcf6e8e03eac2efcd8c407e2317dca5fed26aab91e09'}

## CARD 09 — 극단값과 경계 사례

**삭제나 오류를 함의하지 않는다.** 각 항목에 상태 코드를 붙여 있는 그대로 공개한다.

In [21]:
_bounds = [
    ("TP ≥ 3", "token_premium >= 3", "PLAUSIBLE_EXTREME",
     "짧은 문장의 성긴 비 격자에서 발생한다"),
    ("TP < 0.5", "token_premium < 0.5", "PLAUSIBLE_EXTREME",
     "반대 방향 극단. 역시 짧은 문장에 집중된다"),
    ("어절 수 = 1", "morph_eojeol_count = 1", "EXPECTED_BOUNDARY",
     "제목·항목·짧은 구. 언어적 복잡도 지표가 아니다"),
    ("번역 방향 UNKNOWN", "translation_direction = 'UNKNOWN'", "EXPECTED_BOUNDARY",
     "정책상 보존. 제거하지 않는다"),
    ("조사 비율 = 0", "particle_ratio = 0", "EXPECTED_BOUNDARY",
     "명사구·제목에서 자연스럽게 발생하는 0-질량"),
    ("한국어 측 한글 비율 < 0.1", "ko_hangul_share < 0.1", "REVIEW_REQUIRED",
     "숫자·코드·고유명사 항목 가능성. 결함으로 확정하지 않았다"),
    ("영어 측 한글 비율 > 0.1", "en_hangul_share > 0.1", "POSSIBLE_DEFECT",
     "원문 미확인. 개수가 작아 집계값에 실질 영향 없음"),
]
C9 = {"policy": "no deletion, no error implication", "entries": []}
_rows = []
for _ko, _w, _st, _note in _bounds:
    _n = int(q(f"SELECT count(*) FROM {A} WHERE {_w}")[0])
    _e = {"label_ko": _ko, "predicate": _w, "n": _n, "share": _n / N, "status": _st, "note": _note}
    C9["entries"].append(_e)
    _rows.append({"경계 사례": _ko, "n": _n, "share(%)": round(_n / N * 100, 4),
                  "상태": _st, "설명": _note})
    stat("CARD09", _ko, f"boundary_{_w}", _n / N, "DESCRIPTIVE_PERSISTED",
         source_artifact="D-01…D-05", source_sha=D04_SHA, n=N,
         numerator=f"{_n:,}", denominator=f"{N:,}", note=f"{_st} — {_note}")
show(pd.DataFrame(_rows))
print("\n삭제된 행: 0 — 어떤 경계 사례도 cohort에서 제거되지 않았다")
C9["rows_deleted"] = 0

claim("C-12", "극단적으로 보이는 사례들은 대부분 짧은 문장에서 생기는 정상적 경계이며, "
              "어느 것도 삭제되지 않았다.",
      "TP ≥ 3: n=%d; TP < 0.5: n=%d; eojeol=1: n=%d. 상태 코드로 분류하되 결과 의존적 "
      "사후 행 삭제는 금지 (SSOT §16.3)." % (C9["entries"][0]["n"], C9["entries"][1]["n"],
                                            C9["entries"][2]["n"]),
      "DESCRIPTIVE_PERSISTED", n=N, estimate="rows deleted = 0",
      ci_or_range="—", numerator="0", denominator=f"{N:,}",
      source_artifact="D-01…D-05", source_SHA=D04_SHA, audit_SHA="",
      allowed_wording="경계 사례를 개수·비율·상태와 함께 공개",
      forbidden_wording="이상치를 제거한 뒤의 수치 / 이 사례들은 오류다")

say("CARD09",
    "극단 사례는 개수와 비율을 그대로 공개했고, 383만 문장쌍 중 어느 하나도 분석에서 "
    "제외하지 않았다.",
    "이 사례들은 데이터 오류다 / 제거하면 더 정확한 수치가 나온다 — 원문을 확인하지 않았고 "
    "결함으로 확정된 항목은 없다.")

            경계 사례      n  share(%)                상태                               설명
           TP ≥ 3   3647    0.0951 PLAUSIBLE_EXTREME            짧은 문장의 성긴 비 격자에서 발생한다
         TP < 0.5   1203    0.0314 PLAUSIBLE_EXTREME         반대 방향 극단. 역시 짧은 문장에 집중된다
         어절 수 = 1  42096    1.0974 EXPECTED_BOUNDARY      제목·항목·짧은 구. 언어적 복잡도 지표가 아니다
    번역 방향 UNKNOWN  50547    1.3177 EXPECTED_BOUNDARY                 정책상 보존. 제거하지 않는다
        조사 비율 = 0 292186     7.617 EXPECTED_BOUNDARY         명사구·제목에서 자연스럽게 발생하는 0-질량
한국어 측 한글 비율 < 0.1   1400    0.0365   REVIEW_REQUIRED 숫자·코드·고유명사 항목 가능성. 결함으로 확정하지 않았다
 영어 측 한글 비율 > 0.1     14    0.0004   POSSIBLE_DEFECT     원문 미확인. 개수가 작아 집계값에 실질 영향 없음

삭제된 행: 0 — 어떤 경계 사례도 cohort에서 제거되지 않았다

[CARD09]
  CAN_SAY    : 극단 사례는 개수와 비율을 그대로 공개했고, 383만 문장쌍 중 어느 하나도 분석에서 제외하지 않았다.
  CANNOT_SAY : 이 사례들은 데이터 오류다 / 제거하면 더 정확한 수치가 나온다 — 원문을 확인하지 않았고 결함으로 확정된 항목은 없다.


In [22]:
fig, ax = plt.subplots(figsize=(11.5, 5.2))
_col = {"EXPECTED_BOUNDARY": "#7f9c96", "PLAUSIBLE_EXTREME": "#e8a33d",
        "REVIEW_REQUIRED": "#c1272d", "POSSIBLE_DEFECT": "#8d2f2f"}
_d = pd.DataFrame(C9["entries"]).sort_values("n")
_b = ax.barh(_d["label_ko"], _d["n"], color=[_col[s] for s in _d["status"]])
for _bb, _n2, _s in zip(_b, _d["n"], _d["share"], strict=True):
    ax.text(_n2, _bb.get_y() + _bb.get_height() / 2, f"  {_n2:,}  ({_s * 100:.4f}%)",
            va="center", fontsize=9)
ax.set_xscale("log"); ax.set_xlim(0.7, _d["n"].max() * 60)
ax.set_xlabel("문장쌍 수 (로그 눈금)")
ax.set_title("EP09 · 경계 사례와 극단값 — 상태 코드별\n삭제 없음 · 오류 함의 없음", fontsize=12.5)
_handles = [plt.Rectangle((0, 0), 1, 1, color=v) for v in _col.values()]
ax.legend(_handles, list(_col.keys()), loc="lower right", fontsize=8.5)
fig.tight_layout()
save_fig(fig, "EP09_extremes", "경계 사례와 극단값", card="CARD09", status="DESCRIPTIVE_PERSISTED")

  saved EP09_extremes  sha a6aa4fd26692…  article_ready=NO


{'figure_id': 'EP09_extremes',
 'title_ko': '경계 사례와 극단값',
 'card': 'CARD09',
 'status': 'DESCRIPTIVE_PERSISTED',
 'article_ready': 'NO',
 'png': 'outputs/figures/evidence_pack/EP09_extremes.png',
 'svg': 'outputs/figures/evidence_pack/EP09_extremes.svg',
 'png_sha256': 'a6aa4fd26692f0fe900129fbfebbe1f061d708923e35c95afa0ad3d13295c88f',
 'svg_sha256': '868743175dc1cf519e1a65c3a10e185f59090a571d742d56569c539278529a77'}

## CARD 10 — 설명 blocks (NB09, 감사 통과 후에만)

전제 조건: `NB09_RESULT_AUDIT_PASS`. 이 카드의 모든 수치는 감사된 결과 산출물에서
**그대로 소비**한다 — 모형을 다시 적합하지 않는다.

해석은 **effect size 우선**이다. p-value로 예측자를 서열화하지 않는다.

### 보충감사 반영 — 점추정치와 구간을 분리한다

`NB09 supplemental audit` (`c12ca806…`, canonical main `1b212986…`)은 block ΔR² bootstrap
**구간 구성에 결함**이 있음을 확인했다:
`BOOTSTRAP_BLOCK_CI = NARROW_METHOD_DEFECT_REQUIRING_CI_CORRECTION`.
수정 범위는 **block ΔR² 구간과 `delta_r2_bootstrap_sd` 뿐**이며,
점추정치·R²·partial R²·nested 통계는 영향을 받지 않는다.
선행 `NB09_RESULT_AUDIT_PASS` (`27814139…`)는 그대로 유효하다.

따라서 이 카드는 결과 전체를 강등하지 않고 **두 축으로 분리**한다.

| 축 | 상태 |
|---|---|
| `ARTICLE_READY_POINT_ESTIMATE` | **YES** — R², ΔR², partial R² 점추정치 |
| `ARTICLE_READY_BOOTSTRAP_CI` | **NO** — `NB09_BLOCK_CI_STATUS = CORRECTION_PENDING` |

감사자가 자기 seed로 계산한 대조 구간은 **결함 크기의 증거이지 수정된 결과가 아니다**.
여기서 그 값을 대체 구간으로 채택하지 않는다. B의 frozen-seed 재계산과 A의 검증이
도착한 뒤에만 구간을 복원한다.

In [23]:
_audit_ok = ("NB09_RESULT_AUDIT_PASS" in
             (ROOT / "ssot" / "2026-08-18_1647_KOEN_TP_NB09_RESULT_INDEPENDENT_AUDIT.md")
             .read_text(encoding="utf-8"))
NB09_INCLUDED = bool(_audit_ok)
print(f"NB09_RESULT_AUDIT_PASS 확인 = {NB09_INCLUDED}   audit SHA = {NB09_AUDIT_SHA}")
if not NB09_INCLUDED:
    raise SystemExit("NB09 audit PASS를 확인하지 못했다 — CARD 10/11을 생성하지 않는다")

_supp_path = ROOT / "ssot" / "2026-08-18_1713_KOEN_TP_NB09_SUPPLEMENTAL_AUDIT.md"
_supp = _supp_path.read_text(encoding="utf-8")
SUPPLEMENTAL = {
    "audit_sha": NB09_SUPPLEMENTAL_AUDIT,
    "document": _supp_path.relative_to(ROOT).as_posix(),
    "document_sha256": sha256_file(_supp_path),
    "finding": "BOOTSTRAP_BLOCK_CI = NARROW_METHOD_DEFECT_REQUIRING_CI_CORRECTION",
    "correction_scope": "block ΔR² bootstrap interval + delta_r2_bootstrap_sd only",
    "point_estimates_affected": False,
    "prior_result_audit_still_valid": "NB09_RESULT_AUDIT_PASS @ " + NB09_AUDIT_SHA,
    "new_hard_fail_count": 0,
    "auditor_diagnostic_intervals": "결함 크기의 증거일 뿐 수정된 결과가 아니다 — 채택하지 않는다",
    "resolution_required": "B의 frozen-seed 재계산 + A의 검증",
}
for _needle, _key in (("NARROW_METHOD_DEFECT_REQUIRING_CI_CORRECTION", "finding_string_present"),
                      ("NEW_HARD_FAIL_COUNT = 0", "no_new_hard_fail_string_present")):
    SUPPLEMENTAL[_key] = _needle in _supp
if not SUPPLEMENTAL["finding_string_present"]:
    raise SystemExit("보충감사 문서에서 block CI 판정 문자열을 찾지 못했다")
print(f"\nNB09 supplemental audit = {NB09_SUPPLEMENTAL_AUDIT}")
print(f"  {SUPPLEMENTAL['finding']}")
print(f"  수정 범위: {SUPPLEMENTAL['correction_scope']}")
print(f"  점추정치 영향: {SUPPLEMENTAL['point_estimates_affected']}  ·  "
      f"새 HARD FAIL: {SUPPLEMENTAL['new_hard_fail_count']}")
print(f"  NB09_BLOCK_CI_STATUS = {NB09_BLOCK_CI_STATUS}  ·  ARTICLE_READY_CI = {ARTICLE_READY_CI}")

_mods = pd.DataFrame([m for m in NB09["models"] if m["outcome"] == "A"])
_modsB = pd.DataFrame([m for m in NB09["models"] if m["outcome"] == "B"])
_blocks = pd.DataFrame(NB09["block_comparisons"])
_bA = _blocks[_blocks["outcome"] == "A"].set_index("comparison")
_bB = _blocks[_blocks["outcome"] == "B"].set_index("comparison")

C10 = {"audit": "NB09_RESULT_AUDIT_PASS", "audit_sha": NB09_AUDIT_SHA,
       "results_sha256": NB09_SHA, "estimator": NB09["estimator"],
       "se_type": NB09["se_type"], "bootstrap_B": NB09["bootstrap"]["B"],
       "cohort_N": NB09["cohort"]["N"]}
print("\nOutcome A (log_token_premium) — 감사된 R² 사다리:")
show(_mods[["model", "p", "r2", "adj_r2"]].rename(
    columns={"model": "모형", "p": "모수 수", "r2": "R²", "adj_r2": "조정 R²"}), "{:,.6f}")
for _, _r in _mods.iterrows():
    C10[f"{_r['model']}_r2_A"] = float(_r["r2"])
    stat("CARD10", f"{_r['model']} 설명력 R² (Outcome A)", f"{_r['model']}_r2_A", float(_r["r2"]),
         "AUDITED_RESULT", source_artifact="NB09_EXPLANATORY_RESULTS_v001",
         source_sha=NB09_SHA, audit_sha=NB09_AUDIT_SHA, n=int(_r["n"]))

_map = {"RQ3": ("표현·표면형 block (M1 − M0)", "M0", "M1"),
        "RQ4": ("형태소 block (M2 − M1)", "M1", "M2"),
        "RQ5": ("regex/tokenizer mechanism block (M3 − M2)", "M2", "M3")}
_rows = []
for _rq, (_ko, _red, _full) in _map.items():
    _b = _bA.loc[_rq]
    _quar = (f"[{_b['delta_r2_bootstrap_ci95'][0]:.6f}, "
             f"{_b['delta_r2_bootstrap_ci95'][1]:.6f}]")
    _rows.append({"비교": _rq, "block": _ko, "축소": _red, "완전": _full,
                  "ΔR² (점추정)": _b["delta_r2"],
                  "ΔR² 구간": NB09_BLOCK_CI_STATUS,
                  "partial R²": _b["partial_r2"], "df": _b["df"]})
    C10[f"{_rq}_delta_r2_A"] = float(_b["delta_r2"])
    C10[f"{_rq}_partial_r2_A"] = float(_b["partial_r2"])
    C10[f"{_rq}_ci_A"] = NB09_BLOCK_CI_STATUS          # 구간은 인용하지 않는다
    C10.setdefault("quarantined_block_ci", {})[_rq] = {
        "withdrawn_interval": _quar,
        "withdrawn_bootstrap_sd": float(_b["delta_r2_bootstrap_sd"]),
        "reason": SUPPLEMENTAL["finding"], "status": NB09_BLOCK_CI_STATUS,
        "article_ready": "NO"}
    stat("CARD10", f"{_ko} ΔR² 점추정 (Outcome A)", f"{_rq}_delta_r2_A", float(_b["delta_r2"]),
         "AUDITED_RESULT", source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_sha=NB09_SHA,
         audit_sha=NB09_AUDIT_SHA, n=int(_b["n"]),
         ci=NB09_BLOCK_CI_STATUS, ci_status="CORRECTION_PENDING", quarantined_ci=_quar,
         note=CI_QUARANTINE_NOTE)
    stat("CARD10", f"{_ko} partial R² (Outcome A)", f"{_rq}_partial_r2_A",
         float(_b["partial_r2"]), "AUDITED_RESULT",
         source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_sha=NB09_SHA,
         audit_sha=NB09_AUDIT_SHA, n=int(_b["n"]),
         note="partial R²는 구간 결함의 영향을 받지 않는다")
print("\nblock 증분 설명력 (Outcome A) — 점추정치만 인용 가능:")
show(pd.DataFrame(_rows), "{:,.6f}")
_sens = _bA.loc["SENS_M2A"]
C10["SENS_M2A_delta_r2_A"] = float(_sens["delta_r2"])
C10["quarantined_block_ci"]["SENS_M2A"] = {
    "withdrawn_interval": f"[{_sens['delta_r2_bootstrap_ci95'][0]:.6f}, "
                          f"{_sens['delta_r2_bootstrap_ci95'][1]:.6f}]",
    "withdrawn_bootstrap_sd": float(_sens["delta_r2_bootstrap_sd"]),
    "reason": SUPPLEMENTAL["finding"], "status": NB09_BLOCK_CI_STATUS, "article_ready": "NO"}
print(f"\n형태소 대안 block M2A (기능형태소 비율): ΔR² 점추정 = {_sens['delta_r2']:.6f} "
      f"· 구간 {NB09_BLOCK_CI_STATUS}")
C10["NB09_BLOCK_CI_STATUS"] = NB09_BLOCK_CI_STATUS
C10["ARTICLE_READY_CI"] = ARTICLE_READY_CI
C10["supplemental_audit"] = SUPPLEMENTAL
C10["provenance"] = {"NB09_RESULT_AUDIT": NB09_RESULT_AUDIT,
                     "NB09_SUPPLEMENTAL_AUDIT": NB09_SUPPLEMENTAL_AUDIT,
                     "CURRENT_CANONICAL_MAIN": CURRENT_CANONICAL_MAIN}
print("\n격리된 block bootstrap 구간 (기록만 하고 인용하지 않는다):")
for _k2, _v2 in C10["quarantined_block_ci"].items():
    print(f"  {_k2:9s} withdrawn {_v2['withdrawn_interval']}  sd {_v2['withdrawn_bootstrap_sd']:.3e}"
          f"  → {_v2['status']}")
print("\n[해석 제약 — 감사된 산출물이 스스로 부과한 것]")
for _k, _v in NB09["claim_restrictions"].items():
    print(f"  {_k}: {_v}")

claim("C-13", "문장의 표현·표면형 구조만으로 프리미엄 변동의 상당 부분이 설명된다.",
      f"RQ3 (M1 − M0) ΔR² = {C10['RQ3_delta_r2_A']:.6f}, partial R² = "
      f"{C10['RQ3_partial_r2_A']:.6f}. Outcome A = log_token_premium, fixed-effects OLS, HC1. "
      f"bootstrap 구간은 {NB09_BLOCK_CI_STATUS}.",
      "AUDITED_RESULT", n=N, estimate=f"ΔR² = {C10['RQ3_delta_r2_A']:.4f}",
      ci_or_range=NB09_BLOCK_CI_STATUS,
      ci_status="CORRECTION_PENDING",
      quarantined_ci=C10["quarantined_block_ci"]["RQ3"]["withdrawn_interval"],
      numerator="—", denominator=f"{N:,}",
      source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_SHA=NB09_SHA,
      audit_SHA=NB09_AUDIT_SHA,
      allowed_wording="표현 block을 더했을 때 설명 R²가 0.62 증가했다",
      forbidden_wording="표현 구조가 프리미엄의 62%를 원인으로 만든다")
claim("C-14", "형태소 지표는 표현 구조를 통제한 뒤에는 추가 설명력이 매우 작다.",
      f"RQ4 (M2 − M1) ΔR² = {C10['RQ4_delta_r2_A']:.6f}, partial R² = "
      f"{C10['RQ4_partial_r2_A']:.6f}. 대안 block M2A도 같은 크기 "
      f"(ΔR² = {C10['SENS_M2A_delta_r2_A']:.6f}). bootstrap 구간은 {NB09_BLOCK_CI_STATUS}.",
      "AUDITED_RESULT", n=N, estimate=f"ΔR² = {C10['RQ4_delta_r2_A']:.6f}",
      ci_or_range=NB09_BLOCK_CI_STATUS,
      ci_status="CORRECTION_PENDING",
      quarantined_ci=C10["quarantined_block_ci"]["RQ4"]["withdrawn_interval"],
      numerator="—", denominator=f"{N:,}",
      source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_SHA=NB09_SHA,
      audit_SHA=NB09_AUDIT_SHA,
      allowed_wording="형태소 block의 증분 설명력은 0.0066으로 작다 (0이 아니지만 작다)",
      forbidden_wording="형태소는 프리미엄과 무관하다 / 한국어 형태소 구조는 영향이 없다")
claim("C-15", "tokenizer 내부 구조 지표를 넣으면 설명력이 크게 오르지만, 이는 결과가 만들어지는 "
              "과정 자체에 가까워 인과 해석 대상이 아니다.",
      f"RQ5 (M3 − M2) ΔR² = {C10['RQ5_delta_r2_A']:.6f}, partial R² = "
      f"{C10['RQ5_partial_r2_A']:.6f}. SSOT MN-02: post-treatment mechanism audit. "
      "RQ5 primary 증거는 block 수준 M3−M2이며 raw chunk-count 계수 해석은 금지. "
      f"bootstrap 구간은 {NB09_BLOCK_CI_STATUS}.",
      "AUDITED_RESULT", n=N, estimate=f"ΔR² = {C10['RQ5_delta_r2_A']:.4f}",
      ci_or_range=NB09_BLOCK_CI_STATUS,
      ci_status="CORRECTION_PENDING",
      quarantined_ci=C10["quarantined_block_ci"]["RQ5"]["withdrawn_interval"],
      numerator="—", denominator=f"{N:,}",
      source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_SHA=NB09_SHA,
      audit_SHA=NB09_AUDIT_SHA,
      allowed_wording="mechanism block이 추가로 설명하는 R²는 0.15다 (post-treatment)",
      forbidden_wording="regex chunking이 프리미엄의 가장 큰 원인이다 / mechanism이 15%를 유발한다")

say("CARD10",
    "표현·표면형 block 추가 시 설명 R²는 0.0286에서 0.6496으로 증가했고(ΔR² = 0.6210), "
    "형태소 block의 추가 설명력은 ΔR² = 0.0066, tokenizer mechanism block은 ΔR² = 0.1508이었다. "
    "각 block의 bootstrap interval은 보충감사에서 구성 결함이 확인되어 수정 중이다.",
    "표현 구조가 프리미엄의 62%의 원인이다 / 형태소는 아무 상관이 없다 / regex가 가장 큰 "
    "원인이다 / 기존에 공표된 bootstrap 구간을 인용하는 것 — 이들은 설명 분산 분해이지 인과 "
    "기여가 아니며, block 구간은 현재 CORRECTION_PENDING 이다.")

NB09_RESULT_AUDIT_PASS 확인 = True   audit SHA = 27814139a43bfbe3d4eb2e646e97c0107acfa582

NB09 supplemental audit = c12ca806bfd8245264bba93b52128a07c71be41a
  BOOTSTRAP_BLOCK_CI = NARROW_METHOD_DEFECT_REQUIRING_CI_CORRECTION
  수정 범위: block ΔR² bootstrap interval + delta_r2_bootstrap_sd only
  점추정치 영향: False  ·  새 HARD FAIL: 0
  NB09_BLOCK_CI_STATUS = CORRECTION_PENDING  ·  ARTICLE_READY_CI = NO

Outcome A (log_token_premium) — 감사된 R² 사다리:
 모형  모수 수       R²    조정 R²
 M0     8 0.028625 0.028623
 M1    23 0.649578 0.649576
 M2    27 0.656169 0.656166
M2A    26 0.655423 0.655421
 M3    45 0.806939 0.806936

block 증분 설명력 (Outcome A) — 점추정치만 인용 가능:
 비교                                     block 축소 완전  ΔR² (점추정)             ΔR² 구간  partial R²  df
RQ3                    표현·표면형 block (M1 − M0) M0 M1   0.620954 CORRECTION_PENDING    0.639252  15
RQ4                       형태소 block (M2 − M1) M1 M2   0.006590 CORRECTION_PENDING    0.018807   4
RQ5 regex/tokenizer mechanism block (M3 − M2) M2 M3   0

## CARD 11 — Outcome B와 `B-N01` (구조적 동치)

감사자가 검증한 사실: **M1 이상에서 Outcome B의 적합은 Outcome A의 적합과 같다.**
두 결과를 "서로를 확증하는 두 연구"로 제시하면 안 된다.

In [24]:
_bn = NB09["outcome_ab_structural_relationship"]["finding"]
C11 = {"id": _bn["id"], "class": _bn["class"], "statement": _bn["statement"],
       "not_a_defect": _bn["not_a_defect"], "measured": _bn["measured"],
       "consequences": _bn["consequences"], "label": "STRUCTURAL_EQUIVALENCE",
       "audit": "A가 §5에서 검증 — 결함 아님", "audit_sha": NB09_AUDIT_SHA}
print(f"[{C11['id']}] {C11['class']}\n{C11['statement']}\n")
for _c in C11["consequences"]:
    print(" •", _c)
print(f"\n측정된 수치: {json.dumps(C11['measured'], ensure_ascii=False)}")

_eq = []
for _rq, (_ko, _red, _full) in _map.items():
    _a, _b2 = _bA.loc[_rq], _bB.loc[_rq]
    _eq.append({"비교": _rq, "block": _ko,
                "A partial R²": _a["partial_r2"], "B partial R²": _b2["partial_r2"],
                "동일?": abs(_a["partial_r2"] - _b2["partial_r2"]) < 1e-12,
                "A ΔR²": _a["delta_r2"], "B ΔR²": _b2["delta_r2"],
                "A R²(완전)": _a["r2_full"], "B R²(완전)": _b2["r2_full"]})
_eqdf = pd.DataFrame(_eq)
C11["partial_r2_comparison"] = _eqdf.to_dict(orient="records")
print("\nA와 B의 partial R² 대조:")
show(_eqdf[["비교", "block", "A partial R²", "B partial R²", "동일?", "A ΔR²", "B ΔR²"]], "{:,.8f}")
_sstA = float(_mods.loc[_mods["model"] == "M1", "sst"].iloc[0])
_sstB = float(_modsB.loc[_modsB["model"] == "M1", "sst"].iloc[0])
C11["sst_A"], C11["sst_B"] = _sstA, _sstB
print(f"\n왜 ΔR²만 다른가: 분자(SSR 감소량)는 같고 분모(SST)만 다르다.")
print(f"  SST(A) = {_sstA:,.3f}   SST(B) = {_sstB:,.3f}   비 = {_sstA / _sstB:.6f}")
print(f"  검산 — RQ4: A ΔR² / B ΔR² = "
      f"{_bA.loc['RQ4', 'delta_r2'] / _bB.loc['RQ4', 'delta_r2']:.6f}  "
      f"(= SST(B)/SST(A) = {_sstB / _sstA:.6f})")
stat("CARD11", "B-N01 구조적 동치 (M1 이상)", "B_N01_structural_equivalence", True,
     "AUDITED_RESULT", source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_sha=NB09_SHA,
     audit_sha=NB09_AUDIT_SHA, n=N,
     note="RQ4·RQ5 partial R²가 두 outcome에서 동일 — 독립 재현이 아니다")

claim("C-16", "두 번째 결과변수(압축 penalty)로 얻은 결과는 첫 번째 결과의 독립 재현이 아니다.",
      "B-N01: M1 이상에서 Outcome B의 적합은 Outcome A의 적합과 동일하며 (잔차·SSR·AIC·BIC "
      "일치), RQ4·RQ5의 partial R²는 구조적으로 같다. ΔR²만 SST 분모 차이로 달라진다. "
      "STRUCTURAL_EQUIVALENCE.", "AUDITED_RESULT", n=N,
      estimate="RQ4·RQ5 partial R² A == B", ci_or_range="max |SSR 상대차| 4.2e-13",
      numerator="—", denominator=f"{N:,}",
      source_artifact="NB09_EXPLANATORY_RESULTS_v001", source_SHA=NB09_SHA,
      audit_SHA=NB09_AUDIT_SHA,
      allowed_wording="두 결과변수는 같은 적합의 두 표현이다 (구조적 동치)",
      forbidden_wording="두 개의 서로 다른 결과변수에서 같은 결론이 확인되었다 (독립 재현)")

say("CARD11",
    "압축 penalty를 결과변수로 삼은 분석은 주 결과변수 분석과 수학적으로 같은 적합이며, "
    "형태소·mechanism block의 partial R²는 구조적으로 동일하다.",
    "서로 다른 두 결과변수에서 결과가 재현되었다 / 두 분석이 서로를 확증한다 — 같은 적합의 "
    "두 표현이므로 독립적 확증이 아니다.")

[B-N01] STRUCTURAL_CONSEQUENCE_OF_THE_APPROVED_COMMON_LADDER
For every model containing both log_code_point_ratio and log_byte_density_ratio (M1, M2, M2A, M3), the Outcome B fit is the Outcome A fit with those two coefficients shifted by exactly -1 and every other coefficient unchanged. The residual vector, and therefore SSR, AIC, BIC and every nested test statistic, are identical between the two outcomes.

 • RQ4 and RQ5 partial R-squared are IDENTICAL across the two outcomes by construction; they are not two independent confirmations.
 • Delta R-squared differs between outcomes only through the SST denominator, because the numerator SSR difference is the same quantity.
 • AIC and BIC coincide for M1, M2, M2A and M3; only M0 differs, because M0 omits the two decomposition components.
 • Only RQ3 (M1 - M0) carries genuinely different information between the outcomes.

측정된 수치: {"max_abs_other_coef_difference": 8.280709451469193e-12, "max_ssr_relative_difference": 4.1959943135679447e-13,

In [25]:
fig, axes = plt.subplots(1, 3, figsize=(17.0, 5.0))
_order = ["M0", "M1", "M2", "M3"]
_r2 = [C10[f"{m}_r2_A"] for m in _order]
_b = axes[0].bar(_order, _r2, color=["#b9c6cc", "#3a7ca5", "#2f6690", "#16425b"])
for _bb, _v in zip(_b, _r2, strict=True):
    axes[0].text(_bb.get_x() + _bb.get_width() / 2, _v + 0.012, f"{_v:.4f}", ha="center", fontsize=10)
axes[0].set_ylim(0, 0.92); axes[0].set_ylabel("설명력 R² (Outcome A)")
axes[0].set_title("(가) 모형 사다리 R²\nAUDITED_RESULT · NB09")

_lab = ["표현·표면형\n(M1−M0)", "형태소\n(M2−M1)", "mechanism\n(M3−M2)"]
_dv = [C10["RQ3_delta_r2_A"], C10["RQ4_delta_r2_A"], C10["RQ5_delta_r2_A"]]
_b2 = axes[1].bar(_lab, _dv, color=["#3a7ca5", "#e8a33d", "#c1272d"])
for _bb, _v in zip(_b2, _dv, strict=True):
    axes[1].text(_bb.get_x() + _bb.get_width() / 2, _v + 0.012, f"{_v:.4f}", ha="center", fontsize=10)
axes[1].set_ylabel("증분 설명력 ΔR² (점추정치)")
axes[1].set_title("(나) block 증분 설명력 — 점추정치만\n"
                  "구간은 보충감사로 CORRECTION_PENDING (오차막대 표시하지 않음)")
axes[1].set_ylim(0, 0.70)
axes[1].text(0.97, 0.62, "bootstrap 구간 미표시\nNB09_BLOCK_CI_STATUS\n= CORRECTION_PENDING",
             transform=axes[1].transAxes, ha="right", va="top", fontsize=8.5,
             bbox={"facecolor": "#fdf0f0", "edgecolor": "#c1272d", "boxstyle": "round,pad=0.4"})

_x = np.arange(3); _w = 0.36
_pa = [_eqdf.loc[i, "A partial R²"] for i in range(3)]
_pb = [_eqdf.loc[i, "B partial R²"] for i in range(3)]
axes[2].bar(_x - _w / 2, _pa, _w, color="#16425b", label="Outcome A (log TP)")
axes[2].bar(_x + _w / 2, _pb, _w, color="#8d6a9f", label="Outcome B (log CP)")
for _i in range(3):
    axes[2].text(_i, max(_pa[_i], _pb[_i]) + 0.02,
                 "동일" if _eqdf.loc[_i, "동일?"] else "다름", ha="center", fontsize=9.5,
                 color="#c1272d" if _eqdf.loc[_i, "동일?"] else "#0f4c81", fontweight="bold")
axes[2].set_xticks(_x, ["RQ3\n표현", "RQ4\n형태소", "RQ5\nmechanism"])
axes[2].set_ylabel("partial R²"); axes[2].set_ylim(0, 0.78)
axes[2].set_title("(다) B-N01 · STRUCTURAL_EQUIVALENCE\nRQ4·RQ5는 두 outcome에서 동일 — 독립 재현 아님")
axes[2].legend(fontsize=8.5)
fig.suptitle(f"EP10 · 설명 block과 Outcome B 구조적 동치 (NB09 감사 통과, N = {N:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "EP10_explanatory_blocks", "설명 block과 구조적 동치",
         card="CARD10/CARD11", status="AUDITED_RESULT")

  saved EP10_explanatory_blocks  sha e077c2205f6e…  article_ready=YES


{'figure_id': 'EP10_explanatory_blocks',
 'title_ko': '설명 block과 구조적 동치',
 'card': 'CARD10/CARD11',
 'status': 'AUDITED_RESULT',
 'article_ready': 'YES',
 'png': 'outputs/figures/evidence_pack/EP10_explanatory_blocks.png',
 'svg': 'outputs/figures/evidence_pack/EP10_explanatory_blocks.svg',
 'png_sha256': 'e077c2205f6e8ea63e8f3b1103c1d0bbf0e2e87b10cc950c138133a456c4329f',
 'svg_sha256': '2307de3e0db8527d64501a3379628a6cad2370ebbbcd8df5ff2f3a3860c3dea3'}

## 아직 말할 수 없는 것 — NB11 robustness · NB10 predictive

두 단계는 아직 산출되지 않았다. 관련 주장은 전부 `NOT_YET_ESTABLISHED` 이며
`ARTICLE_READY = NO` 다. 이 팩은 NB11 · NB10 감사 통과 후 같은 파일에서 갱신된다.

In [26]:
NB11_INCLUDED = (ROOT / "outputs" / "reports" / "NB11_ROBUSTNESS_v001.json").exists()
NB10_INCLUDED = (ROOT / "outputs" / "reports" / "NB10_PREDICTIVE_v001.json").exists()
print(f"NB11_INCLUDED = {NB11_INCLUDED}   NB10_INCLUDED = {NB10_INCLUDED}")
for _cid, _plain, _tech in (
        ("C-17", "이 결과가 다른 모형 설정에서도 흔들리지 않는지는 아직 말할 수 없다.",
         "ROBUSTNESS CLAIM MATRIX (primary estimate · sensitivity range · direction stability · "
         "material attenuation · specification sensitivity) requires NB11 audit PASS."),
        ("C-18", "이 모형이 보지 않은 문장을 얼마나 잘 예측하는지는 아직 말할 수 없다.",
         "PREDICTIVE_GENERALIZATION (MAE · RMSE · holdout R² · split policy · LR-01) requires "
         "PRE-NB10 split manifest and NB10 audit PASS.")):
    claim(_cid, _plain, _tech, "NOT_YET_ESTABLISHED", n=None, estimate="—", ci_or_range="—",
          numerator="—", denominator="—", source_artifact="—", source_SHA="", audit_SHA="",
          allowed_wording="아직 산출되지 않았다고 명시",
          forbidden_wording="설명 R²를 예측 성능처럼 인용 / 민감도 분석 없이 '견고하다'고 서술")
say("NB11 · NB10",
    "설명 모형의 R²와 증분 설명력은 감사를 통과했다.",
    "이 결과가 견고하다(robust) / 이 모형이 새 문장을 잘 예측한다 — 민감도 행렬(NB11)과 "
    "예측 일반화(NB10)는 아직 존재하지 않는다. 설명 R²는 예측 성능이 아니다.")

NB11_INCLUDED = False   NB10_INCLUDED = False

[NB11 · NB10]
  CAN_SAY    : 설명 모형의 R²와 증분 설명력은 감사를 통과했다.
  CANNOT_SAY : 이 결과가 견고하다(robust) / 이 모형이 새 문장을 잘 예측한다 — 민감도 행렬(NB11)과 예측 일반화(NB10)는 아직 존재하지 않는다. 설명 R²는 예측 성능이 아니다.


## CARD 12 — 형태소 기술통계 (인과 해석 금지)

§5F가 요구하는 형태소 주변 분포다. **인과 설명으로 전환하지 않는다.**
형태소 block의 조건부 증분 설명력은 CARD 10의 RQ4에 있고, 그것도 인과가 아니다.

In [27]:
C12 = {"scope": "descriptive marginal distributions only; no causal conversion",
       "analyzer": "Kiwi (D-03) — 특정 버전 산출이며 언어학적 정답이 아니다"}
_mcols = [("morpheme_density", "형태소 밀도 (어절당 형태소 수)"),
          ("particle_ratio", "조사 비율"),
          ("ending_ratio", "어미 비율"),
          ("deriv_affix_ratio", "파생접사 비율"),
          ("function_morpheme_ratio", "기능형태소 비율")]
_rows = []
for _c, _ko in _mcols:
    _s = quantiles(_c)
    _z = int(q(f"SELECT count(*) FROM {A} WHERE {_c} = 0")[0])
    _hi = float(np.quantile(qdf(f"SELECT {_c} FROM {A}")[_c].to_numpy(np.float64), 0.999))
    _ex = int(q(f"SELECT count(*) FROM {A} WHERE {_c} > {_hi}")[0])
    C12[_c] = {**_s, "zero_n": _z, "zero_share": _z / N,
               "p999_threshold": _hi, "above_p999_n": _ex, "above_p999_share": _ex / N}
    _rows.append({"지표": _ko, "평균": _s["mean"], "5%": _s["q05"], "25%": _s["q25"],
                  "중앙값": _s["q50"], "75%": _s["q75"], "95%": _s["q95"],
                  "0 비율": _z / N, "99.9분위 초과 비율": _ex / N})
    stat("CARD12", f"{_ko} 중앙값", f"{_c}_median", _s["q50"], "DESCRIPTIVE_PERSISTED",
         source_artifact="D-03 MORPH_FEATURES_KIWI_v001", source_sha=D03_SHA, n=N)
    stat("CARD12", f"{_ko} 0 비율", f"{_c}_zero_share", _z / N, "DESCRIPTIVE_PERSISTED",
         source_artifact="D-03 MORPH_FEATURES_KIWI_v001", source_sha=D03_SHA, n=N,
         numerator=f"{_z:,}", denominator=f"{N:,}")
show(pd.DataFrame(_rows))
print("\n※ 이 표는 주변 분포다. log TP와의 이변량 연관은 제시하지 않는다 — 통제 없는 연관은")
print("  RQ4의 질문이 아니며, 조건부 증분 설명력은 CARD 10 (RQ4, ΔR² = %.6f)에 있다."
      % C10["RQ4_delta_r2_A"])

claim("C-19", "한국어 문장은 어절 하나당 형태소가 중앙값 2.2개이며, 조사가 하나도 없는 문장도 "
              "7.6%나 된다.",
      "morpheme_density median = %.5f (5–95%% [%.5f, %.5f]); particle_ratio = 0 in %.4f%% of "
      "pairs. Kiwi 특정 버전 산출, 주변 분포만." % (
          C12["morpheme_density"]["q50"], C12["morpheme_density"]["q05"],
          C12["morpheme_density"]["q95"], C12["particle_ratio"]["zero_share"] * 100),
      "DESCRIPTIVE_PERSISTED", n=N,
      estimate=f"형태소 밀도 중앙값 {C12['morpheme_density']['q50']:.4f}",
      ci_or_range=f"5–95% [{C12['morpheme_density']['q05']:.4f}, "
                  f"{C12['morpheme_density']['q95']:.4f}]",
      numerator="—", denominator=f"{N:,}",
      source_artifact="D-03 MORPH_FEATURES_KIWI_v001", source_SHA=D03_SHA, audit_SHA="",
      allowed_wording="형태소 지표의 분포 요약 (Kiwi 특정 버전 기준)",
      forbidden_wording="한국어 형태소 구조가 token 프리미엄을 만든다 / 교착어라서 token이 많다")

say("CARD12",
    "한국어 문장은 어절 하나가 중앙값 2.2개의 형태소로 분석되고, 조사 비율·어미 비율 같은 "
    "지표는 0에 상당한 질량을 갖는 유계 비율 변수다.",
    "교착어 형태소 구조가 token 프리미엄의 원인이다 — 형태소 block의 조건부 증분 설명력은 "
    "ΔR² = 0.0066으로 작고, 그조차 인과가 아니다.")

                지표        평균        5%      25%       중앙값      75%      95%      0 비율  99.9분위 초과 비율
형태소 밀도 (어절당 형태소 수)   2.28848   1.77273        2   2.21429  2.46667        3         0   0.000236184
             조사 비율  0.152412         0 0.115385  0.157895      0.2     0.25 0.0761697   0.000803704
             어미 비율   0.18298 0.0769231 0.133333  0.175439 0.222222 0.318182 0.0172029   0.000136601
           파생접사 비율 0.0638881         0        0 0.0652174      0.1 0.152174   0.27168   0.000565695
          기능형태소 비율  0.335392       0.2 0.292683  0.342105  0.38806     0.45 0.0102258   0.000890253

※ 이 표는 주변 분포다. log TP와의 이변량 연관은 제시하지 않는다 — 통제 없는 연관은
  RQ4의 질문이 아니며, 조건부 증분 설명력은 CARD 10 (RQ4, ΔR² = 0.006590)에 있다.

[CARD12]
  CAN_SAY    : 한국어 문장은 어절 하나가 중앙값 2.2개의 형태소로 분석되고, 조사 비율·어미 비율 같은 지표는 0에 상당한 질량을 갖는 유계 비율 변수다.
  CANNOT_SAY : 교착어 형태소 구조가 token 프리미엄의 원인이다 — 형태소 block의 조건부 증분 설명력은 ΔR² = 0.0066으로 작고, 그조차 인과가 아니다.


## 릴리스 부채 (release debt) — 보고에서 제외하는 항목

아래 두 가지는 **기술 근거 팩을 막지 않지만**, 이 팩에서 실질 수치로 보고하지 않는다.

In [28]:
RELEASE_DEBT = {
    "P_VALUE_UNDERFLOW_METADATA_PENDING_NB13": {
        "issue": "nested/Wald p-value가 부동소수점 하한에서 0.0으로 저장되어 있다",
        "rule": "p = 0 을 실질 수치로 보고하지 않는다",
        "reporting_form": "p < 1e-300 형태로만, 그리고 효과 크기와 함께",
        "owner": "NB13 release stage",
        "blocks_descriptive_pack": False,
    },
    "BH_FDR_PENDING_NB13": {
        "issue": "다중 비교 보정(Benjamini-Hochberg FDR)이 아직 적용되지 않았다",
        "rule": "보정 전 p-value로 예측자를 서열화하지 않는다",
        "owner": "NB13 release stage",
        "blocks_descriptive_pack": False,
    },
}
_p0 = int((pd.DataFrame(NB09["block_comparisons"])["nested_p_value"] == 0.0).sum())
RELEASE_DEBT["P_VALUE_UNDERFLOW_METADATA_PENDING_NB13"]["nested_p_zero_entries"] = _p0
for _k, _v in RELEASE_DEBT.items():
    print(f"[{_k}]")
    for _kk, _vv in _v.items():
        print(f"    {_kk}: {_vv}")
print("\n두 항목 모두 기술 근거 팩의 진행을 막지 않는다 (blocks_descriptive_pack = False).")
say("RELEASE DEBT",
    "효과 크기(ΔR², partial R², 중앙 프리미엄)를 중심으로 보고한다.",
    "p = 0 을 실질 수치로 인용하거나, 다중비교 보정 전 p-value로 예측자의 중요도를 "
    "서열화하는 것 — 둘 다 NB13 release 단계의 미해결 부채다.")

[P_VALUE_UNDERFLOW_METADATA_PENDING_NB13]
    issue: nested/Wald p-value가 부동소수점 하한에서 0.0으로 저장되어 있다
    rule: p = 0 을 실질 수치로 보고하지 않는다
    reporting_form: p < 1e-300 형태로만, 그리고 효과 크기와 함께
    owner: NB13 release stage
    blocks_descriptive_pack: False
    nested_p_zero_entries: 8
[BH_FDR_PENDING_NB13]
    issue: 다중 비교 보정(Benjamini-Hochberg FDR)이 아직 적용되지 않았다
    rule: 보정 전 p-value로 예측자를 서열화하지 않는다
    owner: NB13 release stage
    blocks_descriptive_pack: False

두 항목 모두 기술 근거 팩의 진행을 막지 않는다 (blocks_descriptive_pack = False).

[RELEASE DEBT]
  CAN_SAY    : 효과 크기(ΔR², partial R², 중앙 프리미엄)를 중심으로 보고한다.
  CANNOT_SAY : p = 0 을 실질 수치로 인용하거나, 다중비교 보정 전 p-value로 예측자의 중요도를 서열화하는 것 — 둘 다 NB13 release 단계의 미해결 부채다.


## 갱신 대기 구획 (append-only) — 값을 지어내지 않는다

아래 세 구획은 **비어 있는 상태로 예약**된다. 해당 감사가 통과하면 같은 파일에서 채운다.

In [29]:
PENDING_SECTIONS = {
    "NB09_BLOCK_BOOTSTRAP_CI_CORRECTION": {
        "status": "CORRECTION_PENDING",
        "trigger": "B의 frozen-seed 재계산 + A의 검증",
        "on_arrival": ["CARD10", "EP10", "CURRENT_CLAIM_MATRIX",
                       "ARTICLE_EVIDENCE_CARDS", "CURRENT_EVIDENCE_FOR_REPORTING"],
        "then_set": "ARTICLE_READY_BOOTSTRAP_CI = YES",
        "values": None,
    },
    "NB11_ROBUSTNESS_RANGE": {
        "status": "NOT_YET_ESTABLISHED",
        "trigger": "NB11 audit PASS",
        "planned_columns": ["claim", "primary_estimate", "sensitivity_range",
                            "direction_stability", "material_attenuation",
                            "specification_sensitivity"],
        "planned_figure": "EP11_robustness",
        "values": None,
    },
    "NB10_PREDICTIVE_GENERALIZATION": {
        "status": "NOT_YET_ESTABLISHED",
        "trigger": "PRE-NB10 split manifest + NB10 audit PASS",
        "planned_columns": ["model", "MAE", "RMSE", "holdout_R2", "holdout_N",
                            "split_policy", "LR-01_pass"],
        "planned_figure": "EP12_prediction",
        "separation_rule": "예측 결과는 설명 결과와 절대 섞지 않는다; feature importance는 "
                           "예측 기여일 뿐 인과 효과가 아니다",
        "values": None,
    },
}
for _k, _v in PENDING_SECTIONS.items():
    print(f"[{_k}] status = {_v['status']}   trigger = {_v['trigger']}")
    print(f"    values = {_v['values']}  (지어내지 않는다)")

[NB09_BLOCK_BOOTSTRAP_CI_CORRECTION] status = CORRECTION_PENDING   trigger = B의 frozen-seed 재계산 + A의 검증
    values = None  (지어내지 않는다)
[NB11_ROBUSTNESS_RANGE] status = NOT_YET_ESTABLISHED   trigger = NB11 audit PASS
    values = None  (지어내지 않는다)
[NB10_PREDICTIVE_GENERALIZATION] status = NOT_YET_ESTABLISHED   trigger = PRE-NB10 split manifest + NB10 audit PASS
    values = None  (지어내지 않는다)


## CURRENT_CLAIM_MATRIX — 지금 말할 수 있는 것의 전체 목록

각 주장은 등급과 함께 `allowed_wording` · `forbidden_wording` 을 갖는다.
`article_ready = YES` 는 `AUDITED_CANONICAL` · `AUDITED_RESULT` 에만 붙는다.

In [30]:
CLAIM_DF = pd.DataFrame(CLAIMS)
CARD_DF = pd.DataFrame(CARDS)
SAY_DF = pd.DataFrame(SAY)

print("CURRENT_CLAIM_MATRIX")
show(CLAIM_DF[["claim_id", "status", "article_ready_point_estimate",
               "article_ready_bootstrap_ci", "estimate", "plain_korean_claim"]]
     .assign(plain_korean_claim=lambda d: d["plain_korean_claim"].str.slice(0, 44) + "…"))

_by_status = CLAIM_DF["status"].value_counts().to_dict()
_card_by_status = CARD_DF["status"].value_counts().to_dict()
AUDITED_CLAIMS_COUNT = int((CLAIM_DF["article_ready_point_estimate"] == "YES").sum())
CI_PENDING_CLAIMS = CLAIM_DF.loc[CLAIM_DF["ci_status"] == "CORRECTION_PENDING",
                                 "claim_id"].tolist()
CI_PENDING_CARDS = CARD_DF.loc[CARD_DF["ci_status"] == "CORRECTION_PENDING",
                               "metric"].tolist()
ARTICLE_READY_CARDS = sorted({r["card"] for r in FIGURES if r["article_ready"] == "YES"} |
                             {r["card"] for r in CARDS if r["article_ready"] == "YES"})
print(f"\n주장 등급 분포: {json.dumps(_by_status, ensure_ascii=False)}")
print(f"카드 항목 등급 분포: {json.dumps(_card_by_status, ensure_ascii=False)}")
print(f"\nARTICLE_READY 주장 수 = {AUDITED_CLAIMS_COUNT} / {len(CLAIM_DF)}")
print(f"ARTICLE_READY 근거를 담은 카드 = {ARTICLE_READY_CARDS}")
print(f"CAN_SAY / CANNOT_SAY 쌍 = {len(SAY_DF)}")
print(f"\nARTICLE_READY_POINT_ESTIMATE = YES 인 주장 : {AUDITED_CLAIMS_COUNT}")
print(f"ARTICLE_READY_BOOTSTRAP_CI = NO 로 격리된 주장 : {CI_PENDING_CLAIMS}")
print(f"구간이 격리된 카드 항목 수 : {len(CI_PENDING_CARDS)}")
print(f"NB09_BLOCK_CI_STATUS = {NB09_BLOCK_CI_STATUS}   ARTICLE_READY_CI = {ARTICLE_READY_CI}")

print("\n[중요] 등급 규칙의 실제 귀결")
print("  기술통계 카드(CARD 02~09)는 감사된 canonical artifact에서 계산했지만,")
print("  그 값 자체는 아직 Gate 감사를 받지 않았으므로 DESCRIPTIVE_PERSISTED 이고")
print("  ARTICLE_READY = NO 다. 승격하려면 NB07 descriptive package에 대한")
print("  Claude-B의 science-scope audit이 필요하다.")

CURRENT_CLAIM_MATRIX
claim_id                status article_ready_point_estimate article_ready_bootstrap_ci                     estimate                            plain_korean_claim
    C-01     AUDITED_CANONICAL                          YES                        N/A median TP = 1.3333 (+33.33%) 이 표본에서 o200k_base 기준 한국어 문장의 중앙 token premiu…
    C-02     AUDITED_CANONICAL                          YES                        N/A                     0.879850         문장쌍 10개 중 약 9개에서 한국어 쪽 token이 더 많았다.…
    C-03 DESCRIPTIVE_PERSISTED                           NO                        N/A                median ΔT = 5        중앙 문장쌍에서 한국어가 영어보다 5개 더 많은 token을 썼다.…
    C-04 DESCRIPTIVE_PERSISTED                           NO                        N/A                KO 21 / EN 16 이 표본에서 한국어 문장은 중앙값 21개, 영어 문장은 중앙값 16개의 toke…
    C-05 DESCRIPTIVE_PERSISTED                           NO                        N/A      median log CR = -0.7605 한국어의 token 프리미엄은 '글자가 많아서'가 아니다 — 글자 수는 오히려 …
    C-0

In [31]:
CARD_PATH = TBLDIR / "ARTICLE_EVIDENCE_CARDS_v001.csv"
CLAIM_PATH = TBLDIR / "CURRENT_CLAIM_MATRIX_v001.csv"
DESC_PATH = TBLDIR / "DESCRIPTIVE_STATISTICS_ARTICLE_v001.csv"
CARD_DF.to_csv(CARD_PATH, index=False, encoding="utf-8-sig")
CLAIM_DF.to_csv(CLAIM_PATH, index=False, encoding="utf-8-sig")
DESC_DF = CARD_DF[CARD_DF["status"] == "DESCRIPTIVE_PERSISTED"][
    ["card", "metric", "metric_ko", "value", "n", "numerator", "denominator",
     "source_artifact", "source_sha", "note"]]
DESC_DF.to_csv(DESC_PATH, index=False, encoding="utf-8-sig")

FINISHED_KST = dt.datetime.now(KST).isoformat(timespec="seconds")
REPORT = {
    "artifact_id": "CURRENT_EVIDENCE_FOR_REPORTING_v001",
    "run_id": RUN_ID, "created_kst": FINISHED_KST, "started_kst": STARTED_KST,
    "purpose": "article / broadcast / report / executive briefing 용 정량 근거 팩",
    "lane": "SUPPORTING evidence — 새 canonical 연구 단계가 아니다",
    "base_main_sha": BASE_MAIN_SHA,
    "environment": ENV, "korean_plot_font": KOREAN_PLOT_FONT,
    "frozen_artifacts": ARTIFACTS, "artifact_identity": ARTIFACT_IDENTITY,
    "audited_sources": AUDITED_SOURCES,
    "nb07_descriptive_package": NB07_CANDIDATE,
    "cohort": {"N": N, "distinct_pair_id": NDIST, "pair_set_hash": PAIR_SET},
    "evidence_status_system": {
        "statuses": sorted(VALID_STATUSES),
        "article_ready_statuses": sorted(ARTICLE_READY_STATUSES),
        "rule": "감사받지 않은 결과를 조용히 승격시키지 않는다",
    },
    "provenance": {
        "CURRENT_CANONICAL_MAIN": CURRENT_CANONICAL_MAIN,
        "CREATED_FROM_MAIN": BASE_MAIN_SHA,
        "NB09_RESULT_AUDIT": NB09_RESULT_AUDIT,
        "NB09_SUPPLEMENTAL_AUDIT": NB09_SUPPLEMENTAL_AUDIT,
        "G5_INDEPENDENT_AUDIT": G5_AUDIT_SHA,
    },
    "nb09_block_ci": {
        "NB09_BLOCK_CI_STATUS": NB09_BLOCK_CI_STATUS,
        "ARTICLE_READY_CI": ARTICLE_READY_CI,
        "note": CI_QUARANTINE_NOTE,
        "supplemental_audit": SUPPLEMENTAL,
        "quarantined_intervals": C10["quarantined_block_ci"],
        "auditor_diagnostic_intervals_adopted": False,
    },
    "release_debt": RELEASE_DEBT,
    "pending_sections": PENDING_SECTIONS,
    "cards": {"CARD01": C1, "CARD02": C2, "CARD03": C3, "CARD04": C4, "CARD05": C5,
              "CARD06": C6, "CARD07": C7, "CARD08": C8, "CARD09": C9,
              "CARD10": C10, "CARD11": C11, "CARD12": C12},
    "claim_matrix": CLAIMS,
    "evidence_card_rows": CARDS,
    "can_say_cannot_say": SAY,
    "figures": FIGURES,
    "inclusion": {"NB09_INCLUDED": NB09_INCLUDED, "NB11_INCLUDED": NB11_INCLUDED,
                  "NB10_INCLUDED": NB10_INCLUDED,
                  "NB09_audit_sha": NB09_AUDIT_SHA,
                  "NB11_note": "NB11 robustness 미산출 — ROBUSTNESS CLAIM MATRIX는 감사 통과 후 추가",
                  "NB10_note": "NB10 predictive 미산출 — PREDICTIVE_GENERALIZATION은 감사 통과 후 별도 절"},
    "counts": {"claims": len(CLAIMS), "article_ready_claims": AUDITED_CLAIMS_COUNT,
               "evidence_card_rows": len(CARDS), "figures": len(FIGURES),
               "can_say": len(SAY), "cannot_say": len(SAY),
               "claims_by_status": _by_status, "card_rows_by_status": _card_by_status,
               "claims_with_ci_correction_pending": len(CI_PENDING_CLAIMS),
               "card_rows_with_ci_correction_pending": len(CI_PENDING_CARDS)},
    "privacy": {"raw_text_committed": False, "pair_id_list": False,
                "recoverable_examples": False, "token_sequences": False},
    "telemetry": {"cohort": COHORT_TELEMETRY},
    "global_forbidden_wording": [
        "한국어는 본질적으로 AI에 비효율적이다",
        "UTF-8이 3 byte라서 token이 3배가 된다",
        "token premium이 추론 성능 저하를 뜻한다",
        "순수 출처 효과 / 순수 도메인 효과",
        "예측 중요도 = 인과 효과",
        "형태소가 프리미엄을 일으킨다",
        "가장 큰 원인 (largest cause)",
    ],
}
_out = {}
for _p, _o in ((REPDIR / "CURRENT_EVIDENCE_FOR_REPORTING_v001.json", REPORT),):
    _p.write_text(json.dumps(_o, indent=2, ensure_ascii=False, sort_keys=True, default=float)
                  + "\n", encoding="utf-8")
    _out[_p.name] = _p
for _p in (CARD_PATH, CLAIM_PATH, DESC_PATH):
    _out[_p.name] = _p
for _n, _p in _out.items():
    print(f"  {_p.relative_to(ROOT)}  {_p.stat().st_size / 1024:.1f} KiB  sha {sha256_file(_p)[:12]}…")

  outputs/reports/CURRENT_EVIDENCE_FOR_REPORTING_v001.json  130.5 KiB  sha d632458eec9d…
  outputs/tables/ARTICLE_EVIDENCE_CARDS_v001.csv  22.2 KiB  sha e6a238c01a90…
  outputs/tables/CURRENT_CLAIM_MATRIX_v001.csv  11.9 KiB  sha f63e59e2e5ee…
  outputs/tables/DESCRIPTIVE_STATISTICS_ARTICLE_v001.csv  12.3 KiB  sha 943ac1ae59cc…


## 검증

실행이 스스로를 검증한다. 하나라도 실패하면 중단한다.

In [32]:
V = {}
V["frozen_artifact_identity"] = {"result": ARTIFACT_IDENTITY,
                                 "pass": all(v["match"] for v in ARTIFACTS.values())}
V["cohort_identity"] = {"N": N, "pair_set_hash": PAIR_SET,
                        "pass": N == EXPECTED_N and PAIR_SET == EXPECTED_PAIR_SET}
V["decomposition_identity"] = {"max_abs_error": C3["identity_max_abs_error_recomputed"],
                               "pass": C3["identity_max_abs_error_recomputed"] < 1e-12}
_bad_svg = [f["figure_id"] for f in FIGURES
            if not any("가" <= c <= "힣" for c in (ROOT / f["svg"]).read_text(encoding="utf-8"))]
V["korean_glyph_smoke"] = {"font": KOREAN_PLOT_FONT, "figures": len(FIGURES),
                           "without_korean": _bad_svg, "pass": not _bad_svg}
_disk = {p.stem for p in FIGDIR.glob("*.png")}
_mani = {f["figure_id"] for f in FIGURES}
V["figure_consistency"] = {"on_disk": len(_disk), "in_report": len(_mani),
                           "only_on_disk": sorted(_disk - _mani),
                           "only_in_report": sorted(_mani - _disk),
                           "pass": _disk == _mani}
_leak = {}
for _n, _p in _out.items():
    _t = _p.read_text(encoding="utf-8")
    _leak[_n] = {"pair_id_key": '"pair_id"' in _t or ",pair_id," in _t,
                 "size_bytes": _p.stat().st_size}
V["no_raw_text_leak"] = {"detail": _leak,
                         "pass": not any(v["pair_id_key"] for v in _leak.values())}
V["status_discipline"] = {
    "unaudited_marked_article_ready": [c["claim_id"] for c in CLAIMS
                                       if c["article_ready"] == "YES"
                                       and c["status"] not in ARTICLE_READY_STATUSES],
    "pass": all(c["article_ready"] == ("YES" if c["status"] in ARTICLE_READY_STATUSES else "NO")
                for c in CLAIMS)}
V["every_claim_has_both_wordings"] = {
    "pass": all(c["allowed_wording"] and c["forbidden_wording"] for c in CLAIMS)}
V["every_card_has_can_cannot"] = {"pairs": len(SAY), "pass": len(SAY) >= 9}

# 격리된 구간이 "인용 가능한" 필드로 새어 나가지 않았는지 검사한다
_withdrawn = {v["withdrawn_interval"] for v in C10["quarantined_block_ci"].values()}
_published_fields = ([str(c["CI_or_range"]) for c in CLAIMS]
                     + [str(c["ci_or_range"]) for c in CARDS]
                     + [s2["CAN_SAY"] for s2 in SAY])
_leaked = [f for f in _published_fields for w in _withdrawn if w in f]
V["withdrawn_ci_not_published"] = {
    "withdrawn_intervals": sorted(_withdrawn), "leaked_into_published_fields": _leaked,
    "pass": not _leaked}
V["ci_status_discipline"] = {
    "correction_pending_claims": CI_PENDING_CLAIMS,
    "all_ci_pending_are_no": all(c["article_ready_bootstrap_ci"] == "NO" for c in CLAIMS
                                 if c["ci_status"] == "CORRECTION_PENDING"),
    "pass": all(c["article_ready_bootstrap_ci"] == "NO" for c in CLAIMS
                if c["ci_status"] == "CORRECTION_PENDING")}
V["point_estimates_retained"] = {
    "rq3": C10["RQ3_delta_r2_A"], "rq4": C10["RQ4_delta_r2_A"], "rq5": C10["RQ5_delta_r2_A"],
    "pass": (abs(C10["RQ3_delta_r2_A"] - 0.620954) < 1e-6
             and abs(C10["RQ4_delta_r2_A"] - 0.006590) < 1e-6
             and abs(C10["RQ5_delta_r2_A"] - 0.150770) < 1e-6)}
V["pending_sections_have_no_invented_values"] = {
    "pass": all(v["values"] is None for v in PENDING_SECTIONS.values())}

_failed = [k for k, v in V.items() if not v.get("pass")]
for _k, _v in V.items():
    print(f"  {'PASS' if _v.get('pass') else 'FAIL'}  {_k}")
if _failed:
    raise SystemExit(f"EVIDENCE_PACK_VALIDATION_FAIL: {_failed}")
REPORT["validation"] = V
(REPDIR / "CURRENT_EVIDENCE_FOR_REPORTING_v001.json").write_text(
    json.dumps(REPORT, indent=2, ensure_ascii=False, sort_keys=True, default=float) + "\n",
    encoding="utf-8")

print("\n" + "=" * 72)
print("CURRENT EVIDENCE PACK — RETURN BLOCK")
print("=" * 72)
print(f"BASE_MAIN_SHA            = {BASE_MAIN_SHA}")
print(f"COHORT_N                 = {N:,}   PAIR_SET_HASH = {PAIR_SET}")
print(f"ARTIFACT_IDENTITY        = {ARTIFACT_IDENTITY}")
print(f"CURRENT_CANONICAL_MAIN   = {CURRENT_CANONICAL_MAIN}")
print(f"NB09_RESULT_AUDIT        = {NB09_RESULT_AUDIT}")
print(f"NB09_SUPPLEMENTAL_AUDIT  = {NB09_SUPPLEMENTAL_AUDIT}")
print(f"AUDITED_CLAIMS_COUNT     = {AUDITED_CLAIMS_COUNT} / {len(CLAIMS)}")
print(f"ARTICLE_READY_POINT_ESTIMATES = YES  ({AUDITED_CLAIMS_COUNT} claims)")
print(f"ARTICLE_READY_BOOTSTRAP_CI    = {ARTICLE_READY_CI}  "
      f"(NB09_BLOCK_CI_STATUS = {NB09_BLOCK_CI_STATUS})")
print(f"  격리된 구간 {len(C10['quarantined_block_ci'])}건 — 기록만 하고 인용하지 않는다")
print(f"ARTICLE_READY_CARDS      = {ARTICLE_READY_CARDS}")
print(f"NB09_INCLUDED            = {'YES' if NB09_INCLUDED else 'NO'}  (audit {NB09_AUDIT_SHA[:12]}…)")
print(f"NB11_INCLUDED            = {'YES' if NB11_INCLUDED else 'NO'}")
print(f"NB10_INCLUDED            = {'YES' if NB10_INCLUDED else 'NO'}")
print(f"CAN_SAY_COUNT            = {len(SAY)}")
print(f"CANNOT_SAY_COUNT         = {len(SAY)}")
print(f"FIGURES                  = {len(FIGURES)} (PNG + SVG)")
for _f in FIGURES:
    print(f"  {_f['figure_id']:32s} {_f['status']:22s} article_ready={_f['article_ready']}")
print("RAW_TEXT_COMMITTED       = NO")
print(f"finished_kst             = {FINISHED_KST}")
print("=" * 72)
print("CURRENT_EVIDENCE_PACK_COMPLETE")

  PASS  frozen_artifact_identity
  PASS  cohort_identity
  PASS  decomposition_identity
  PASS  korean_glyph_smoke
  PASS  figure_consistency
  PASS  no_raw_text_leak
  PASS  status_discipline
  PASS  every_claim_has_both_wordings
  PASS  every_card_has_can_cannot
  PASS  withdrawn_ci_not_published
  PASS  ci_status_discipline
  PASS  point_estimates_retained
  PASS  pending_sections_have_no_invented_values

CURRENT EVIDENCE PACK — RETURN BLOCK
BASE_MAIN_SHA            = bc8187cceae67ac87c9df1c4a08b54ca753ae035
COHORT_N                 = 3,835,988   PAIR_SET_HASH = d9660d654ee449e4d0c23a0070225274
ARTIFACT_IDENTITY        = 5 / 5
CURRENT_CANONICAL_MAIN   = 1b212986527a273301c5fefdf87a41e7fb33dd37
NB09_RESULT_AUDIT        = 27814139a43bfbe3d4eb2e646e97c0107acfa582
NB09_SUPPLEMENTAL_AUDIT  = c12ca806bfd8245264bba93b52128a07c71be41a
AUDITED_CLAIMS_COUNT     = 8 / 19
ARTICLE_READY_POINT_ESTIMATES = YES  (8 claims)
ARTICLE_READY_BOOTSTRAP_CI    = NO  (NB09_BLOCK_CI_STATUS = CORRECTION_PENDI

## 기사·보고서 작성 안전 요약

아래 표가 이 팩의 최종 산출이다. 왼쪽은 **지금 쓸 수 있는 문장**, 오른쪽은 **쓰면 안 되는
문장**이다.

In [33]:
for _s in SAY:
    print(f"\n■ {_s['scope']}")
    print(f"  ○ 쓸 수 있다 : {_s['CAN_SAY']}")
    print(f"  ✕ 쓸 수 없다 : {_s['CANNOT_SAY']}")
print("\n" + "-" * 72)
print("전역 금지 표현:")
for _w in REPORT["global_forbidden_wording"]:
    print("  ✕", _w)


■ CARD01
  ○ 쓸 수 있다 : 이 병렬 말뭉치에서 o200k_base tokenizer 기준 한국어의 중앙 token premium은 33.3%였고, 문장쌍의 87.99%에서 한국어 token 수가 더 많았다.
  ✕ 쓸 수 없다 : 한국어가 본질적으로 AI에 33% 비효율적이다 / 이 수치가 다른 tokenizer·다른 말뭉치·실제 서비스 비용에 그대로 적용된다 / token premium이 추론 성능 저하를 뜻한다.

■ CARD02
  ○ 쓸 수 있다 : 이 표본에서 한국어 문장은 중앙값 21 token, 영어 문장은 중앙값 16 token으로 쪼개졌다.
  ✕ 쓸 수 없다 : 두 중앙값의 비(21/16)를 token premium이라고 부르는 것 — 프리미엄은 문장쌍 내부에서 짝지어 계산한 비의 중앙값이지 주변 요약의 비가 아니다.

■ CARD03
  ○ 쓸 수 있다 : 관측된 token 프리미엄은 '문자 수 비 × UTF-8 byte 밀도 비 × tokenizer 압축 비'로 오차 없이 정확히 분해되며, 한국어는 문자 수에서는 오히려 짧다.
  ✕ 쓸 수 없다 : 세 요소 각각이 프리미엄의 원인을 몇 %씩 만든다 / 성분 중앙값을 곱해서 중앙 프리미엄을 구할 수 있다 / UTF-8이 3 byte라서 token이 3배가 된다.

■ CARD04
  ○ 쓸 수 있다 : 한국어는 문자 수로는 영어보다 짧은 경우가 99.67%인데, 그중에서도 token 수는 오히려 더 많은 경우가 전체의 87.69%다.
  ✕ 쓸 수 없다 : 99.67% · 71.05% · 87.69% 를 모두 같은 '역전' 수치로 섞어 인용하기 — 세 값은 각각 다른 조건을 센다.

■ CARD05
  ○ 쓸 수 있다 : UTF-8 byte 부담과 tokenizer 압축 penalty는 이 표본에서 거의 함께 움직이지 않는 별개의 축이며, 네 조합(높·높 / 높·낮 / 낮·높 / 낮·낮)이 모두 실제로 관측된다.
  ✕ 쓸 수 없다 : 두 지표가 통계적으로 독립이다 / 상관이